## Evaluate Decima's performance on held-out genes and pseudobulks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import anndata
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### Data loading and visualization

In [ ]:
# =============================================================================
# 1. FUNCTIONS TO COMPARE CORRELATIONS BETWEEN DIFFERENT MODEL TYPES
# =============================================================================

def load_multiple_models(model_paths, model_names):
    """
    Load data from multiple model files and combine into a single dataframe
    
    Parameters:
    -----------
    model_paths : list
        List of paths to model .h5ad files
    model_names : list
        List of names for each model (same length as model_paths)
    
    Returns:
    --------
    combined_obs : pd.DataFrame
        Combined observation data with model_type column
    combined_var : pd.DataFrame
        Combined variable data with model_type column
    """
    combined_obs = []
    combined_var = []
    
    for path, name in zip(model_paths, model_names):
        try:
            ad_temp = anndata.read_h5ad(path)
            
            # Add model information to obs
            obs_temp = ad_temp.obs.copy()
            obs_temp['model_type'] = name
            obs_temp['model_file'] = path.split('/')[-1]
            combined_obs.append(obs_temp)
            
            # Add model information to var
            var_temp = ad_temp.var.copy()
            var_temp['model_type'] = name
            var_temp['model_file'] = path.split('/')[-1]
            combined_var.append(var_temp)
            
            print(f"Loaded {name}: {len(obs_temp)} pseudobulks, {len(var_temp)} genes")
            
        except Exception as e:
            print(f"Error loading {path}: {e}")
    
    return pd.concat(combined_obs, ignore_index=True), pd.concat(combined_var, ignore_index=True)

def compare_model_performance(combined_obs, combined_var, datasets=['test', 'train', 'val']):
    """
    Compare performance metrics between different model types
    
    Parameters:
    -----------
    combined_obs : pd.DataFrame
        Combined observation data with model_type column
    combined_var : pd.DataFrame
        Combined variable data with model_type column
    datasets : list
        Which datasets to analyze
    
    Returns:
    --------
    summary_stats : pd.DataFrame
        Summary statistics for each model type
    """
    results = []
    
    # Pseudobulk correlations
    for dataset in datasets:
        col_name = f"{dataset}_pearson"
        if col_name in combined_obs.columns:
            for model_type in combined_obs['model_type'].unique():
                model_data = combined_obs[combined_obs['model_type'] == model_type]
                
                results.append({
                    'model_type': model_type,
                    'dataset': dataset,
                    'metric_type': 'pseudobulk',
                    'mean_correlation': model_data[col_name].mean(),
                    'std_correlation': model_data[col_name].std(),
                    'median_correlation': model_data[col_name].median(),
                    'n_samples': len(model_data),
                    'q25': model_data[col_name].quantile(0.25),
                    'q75': model_data[col_name].quantile(0.75)
                })
    
    # Gene correlations
    for dataset in datasets:
        dataset_var = combined_var[combined_var['dataset'] == dataset]
        if len(dataset_var) > 0:
            for model_type in dataset_var['model_type'].unique():
                model_data = dataset_var[dataset_var['model_type'] == model_type]
                
                results.append({
                    'model_type': model_type,
                    'dataset': dataset,
                    'metric_type': 'gene',
                    'mean_correlation': model_data['pearson'].mean(),
                    'std_correlation': model_data['pearson'].std(),
                    'median_correlation': model_data['pearson'].median(),
                    'n_samples': len(model_data),
                    'q25': model_data['pearson'].quantile(0.25),
                    'q75': model_data['pearson'].quantile(0.75)
                })
    
    return pd.DataFrame(results)

# =============================================================================
# 2. ENHANCED HISTOGRAM PLOTTING FUNCTIONS
# =============================================================================

def plot_correlation_histograms(combined_obs, combined_var, 
                               dataset='test', 
                               figsize=(15, 10),
                               bins=30,
                               alpha=0.35):
    """
    Create comprehensive histogram plots comparing model types
    
    Parameters:
    -----------
    combined_obs : pd.DataFrame
        Combined observation data
    combined_var : pd.DataFrame
        Combined variable data
    dataset : str
        Which dataset to plot ('test', 'train', 'val')
    figsize : tuple
        Figure size
    bins : int
        Number of histogram bins
    alpha : float
        Transparency level
    """
    
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle(f'Correlation Distributions - {dataset.upper()} Dataset', fontsize=16, fontweight='bold')
    
    # Colors for different model types
    model_types = combined_obs['model_type'].unique()
    colors = plt.cm.Set3(np.linspace(0, 1, len(model_types)))
    color_map = dict(zip(model_types, colors))
    
    # Plot 1: Pseudobulk correlations - overlaid histograms
    ax1 = axes[0, 0]
    col_name = f"{dataset}_pearson"
    
    for model_type in model_types:
        model_data = combined_obs[combined_obs['model_type'] == model_type]
        if col_name in model_data.columns:
            ax1.hist(model_data[col_name].dropna(), 
                    bins=bins, alpha=alpha, 
                    label=f'{model_type} (n={len(model_data)})',
                    color=color_map[model_type],
                    density=True)
    
    ax1.set_xlabel('Pearson Correlation')
    ax1.set_ylabel('Density')
    ax1.set_title('Pseudobulk Correlations by Model Type')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Gene correlations - overlaid histograms
    ax2 = axes[0, 1]
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    
    for model_type in model_types:
        model_data = dataset_var[dataset_var['model_type'] == model_type]
        if len(model_data) > 0:
            ax2.hist(model_data['pearson'].dropna(), 
                    bins=bins, alpha=alpha,
                    label=f'{model_type} (n={len(model_data)})',
                    color=color_map[model_type],
                    density=True)
    
    ax2.set_xlabel('Pearson Correlation')
    ax2.set_ylabel('Density')
    ax2.set_title('Gene Correlations by Model Type')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Box plots for pseudobulks
    ax3 = axes[1, 0]
    pseudobulk_data = []
    pseudobulk_labels = []
    
    for model_type in model_types:
        model_data = combined_obs[combined_obs['model_type'] == model_type]
        if col_name in model_data.columns:
            pseudobulk_data.append(model_data[col_name].dropna())
            pseudobulk_labels.append(model_type)
    
    box_plot = ax3.boxplot(pseudobulk_data, labels=pseudobulk_labels, patch_artist=True)
    for patch, color in zip(box_plot['boxes'], [color_map[label] for label in pseudobulk_labels]):
        patch.set_facecolor(color)
        patch.set_alpha(alpha)
    
    ax3.set_ylabel('Pearson Correlation')
    ax3.set_title('Pseudobulk Correlations - Box Plot')
    ax3.grid(True, alpha=0.3)
    ax3.tick_params(axis='x', rotation=45)
    
    # Plot 4: Box plots for genes
    ax4 = axes[1, 1]
    gene_data = []
    gene_labels = []
    
    for model_type in model_types:
        model_data = dataset_var[dataset_var['model_type'] == model_type]
        if len(model_data) > 0:
            gene_data.append(model_data['pearson'].dropna())
            gene_labels.append(model_type)
    
    if gene_data:
        box_plot = ax4.boxplot(gene_data, labels=gene_labels, patch_artist=True)
        for patch, color in zip(box_plot['boxes'], [color_map[label] for label in gene_labels]):
            patch.set_facecolor(color)
            patch.set_alpha(alpha)
    
    ax4.set_ylabel('Pearson Correlation')
    ax4.set_title('Gene Correlations - Box Plot')
    ax4.grid(True, alpha=0.3)
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig

def plot_stratified_histograms(combined_obs, stratify_by='timepoint', 
                              dataset='test', max_categories=8,
                              figsize=(20, 12)):
    """
    Create histograms stratified by a categorical variable
    
    Parameters:
    -----------
    combined_obs : pd.DataFrame
        Combined observation data
    stratify_by : str
        Column name to stratify by
    dataset : str
        Which dataset to analyze
    max_categories : int
        Maximum number of categories to plot
    """
    
    col_name = f"{dataset}_pearson"
    if col_name not in combined_obs.columns:
        print(f"Column {col_name} not found")
        return
    
    # Get top categories by frequency
    categories = combined_obs[stratify_by].value_counts().head(max_categories).index
    
    # Calculate subplot dimensions
    n_cats = len(categories)
    n_models = len(combined_obs['model_type'].unique())
    
    fig, axes = plt.subplots(n_cats, n_models, figsize=figsize, sharey=True)
    if n_cats == 1:
        axes = axes.reshape(1, -1)
    if n_models == 1:
        axes = axes.reshape(-1, 1)
    
    fig.suptitle(f'Pseudobulk Correlations by {stratify_by} and Model Type - {dataset.upper()}', 
                 fontsize=16, fontweight='bold')
    
    colors = plt.cm.Set3(np.linspace(0, 1, n_models))
    
    for i, category in enumerate(categories):
        for j, model_type in enumerate(combined_obs['model_type'].unique()):
            ax = axes[i, j] if n_cats > 1 and n_models > 1 else axes[i] if n_models == 1 else axes[j]
            
            data = combined_obs[
                (combined_obs[stratify_by] == category) & 
                (combined_obs['model_type'] == model_type)
            ][col_name].dropna()
            
            if len(data) > 0:
                ax.hist(data, bins=20, alpha=0.7, color=colors[j], density=True)
                ax.set_title(f'{category}\n{model_type} (n={len(data)})')
                ax.grid(True, alpha=0.3)
            else:
                ax.set_title(f'{category}\n{model_type} (n=0)')
            
            if i == n_cats - 1:
                ax.set_xlabel('Pearson Correlation')
            if j == 0:
                ax.set_ylabel('Density')
    
    plt.tight_layout()
    return fig

# =============================================================================
# 3. STATISTICAL COMPARISON FUNCTIONS
# =============================================================================

def perform_statistical_tests(combined_obs, combined_var, datasets=['test']):
    """
    Perform statistical tests comparing model performance
    
    Parameters:
    -----------
    combined_obs : pd.DataFrame
        Combined observation data
    combined_var : pd.DataFrame
        Combined variable data
    datasets : list
        Which datasets to test
    
    Returns:
    --------
    test_results : pd.DataFrame
        Statistical test results
    """
    
    results = []
    model_types = combined_obs['model_type'].unique()
    
    if len(model_types) < 2:
        print("Need at least 2 model types for comparison")
        return pd.DataFrame()
    
    # Compare pseudobulk correlations
    for dataset in datasets:
        col_name = f"{dataset}_pearson"
        if col_name in combined_obs.columns:
            
            # Prepare data for comparison
            groups = {}
            for model_type in model_types:
                data = combined_obs[combined_obs['model_type'] == model_type][col_name].dropna()
                if len(data) > 0:
                    groups[model_type] = data
            
            # Pairwise comparisons
            model_list = list(groups.keys())
            for i in range(len(model_list)):
                for j in range(i+1, len(model_list)):
                    model1, model2 = model_list[i], model_list[j]
                    data1, data2 = groups[model1], groups[model2]
                    
                    # Mann-Whitney U test (non-parametric)
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = statistic / (len(data1) * len(data2))  # Simple effect size
                        
                        results.append({
                            'dataset': dataset,
                            'metric_type': 'pseudobulk',
                            'model1': model1,
                            'model2': model2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'n1': len(data1),
                            'n2': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in Mann-Whitney test for {model1} vs {model2}: {e}")
                    
                    # Welch's t-test (assuming unequal variances)
                    try:
                        statistic, p_value = stats.ttest_ind(data1, data2, equal_var=False)
                        cohens_d = (data1.mean() - data2.mean()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'dataset': dataset,
                            'metric_type': 'pseudobulk',
                            'model1': model1,
                            'model2': model2,
                            'test': "Welch's t-test",
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': cohens_d,
                            'mean_diff': data1.mean() - data2.mean(),
                            'n1': len(data1),
                            'n2': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in t-test for {model1} vs {model2}: {e}")
    
    # Compare gene correlations
    for dataset in datasets:
        dataset_var = combined_var[combined_var['dataset'] == dataset]
        
        if len(dataset_var) > 0:
            # Prepare data for comparison
            groups = {}
            for model_type in model_types:
                data = dataset_var[dataset_var['model_type'] == model_type]['pearson'].dropna()
                if len(data) > 0:
                    groups[model_type] = data
            
            # Pairwise comparisons
            model_list = list(groups.keys())
            for i in range(len(model_list)):
                for j in range(i+1, len(model_list)):
                    model1, model2 = model_list[i], model_list[j]
                    data1, data2 = groups[model1], groups[model2]
                    
                    # Mann-Whitney U test
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = statistic / (len(data1) * len(data2))
                        
                        results.append({
                            'dataset': dataset,
                            'metric_type': 'gene',
                            'model1': model1,
                            'model2': model2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'n1': len(data1),
                            'n2': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in Mann-Whitney test for {model1} vs {model2}: {e}")
                    
                    # Welch's t-test
                    try:
                        statistic, p_value = stats.ttest_ind(data1, data2, equal_var=False)
                        cohens_d = (data1.mean() - data2.mean()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'dataset': dataset,
                            'metric_type': 'gene',
                            'model1': model1,
                            'model2': model2,
                            'test': "Welch's t-test",
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': cohens_d,
                            'mean_diff': data1.mean() - data2.mean(),
                            'n1': len(data1),
                            'n2': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in t-test for {model1} vs {model2}: {e}")
    
    test_results = pd.DataFrame(results)
    
    # Apply multiple testing correction
    if len(test_results) > 0:
        from statsmodels.stats.multitest import multipletests
        
        for test_type in test_results['test'].unique():
            mask = test_results['test'] == test_type
            if mask.sum() > 1:
                _, p_corrected, _, _ = multipletests(
                    test_results.loc[mask, 'p_value'], 
                    method='fdr_bh'
                )
                test_results.loc[mask, 'p_corrected'] = p_corrected
    
    return test_results

def create_significance_heatmap(test_results, metric_type='pseudobulk', test_type='Mann-Whitney U'):
    """
    Create a heatmap showing significance of pairwise comparisons
    """
    # Filter results
    filtered_results = test_results[
        (test_results['metric_type'] == metric_type) & 
        (test_results['test'] == test_type)
    ]
    
    if len(filtered_results) == 0:
        print(f"No results found for {metric_type} with {test_type}")
        return
    
    # Create matrix
    models = list(set(filtered_results['model1'].tolist() + filtered_results['model2'].tolist()))
    n_models = len(models)
    
    # Initialize matrices
    p_matrix = np.ones((n_models, n_models))
    effect_matrix = np.zeros((n_models, n_models))
    
    model_to_idx = {model: i for i, model in enumerate(models)}
    
    for _, row in filtered_results.iterrows():
        i = model_to_idx[row['model1']]
        j = model_to_idx[row['model2']]
        
        p_val = row['p_corrected'] if 'p_corrected' in row and pd.notna(row['p_corrected']) else row['p_value']
        
        p_matrix[i, j] = p_val
        p_matrix[j, i] = p_val
        effect_matrix[i, j] = row['effect_size']
        effect_matrix[j, i] = -row['effect_size']
    
    # Create plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # P-value heatmap
    im1 = ax1.imshow(-np.log10(p_matrix), cmap='Reds', vmin=0)
    ax1.set_xticks(range(n_models))
    ax1.set_yticks(range(n_models))
    ax1.set_xticklabels(models, rotation=45)
    ax1.set_yticklabels(models)
    ax1.set_title(f'Statistical Significance\n{metric_type} - {test_type}\n(-log10 p-value)')
    
    # Add text annotations
    for i in range(n_models):
        for j in range(n_models):
            if i != j:
                p_val = p_matrix[i, j]
                text = f'{p_val:.3f}'
                if p_val < 0.001:
                    text = '***'
                elif p_val < 0.01:
                    text = '**'
                elif p_val < 0.05:
                    text = '*'
                ax1.text(j, i, text, ha="center", va="center", color="black" if p_val > 0.01 else "white")
    
    plt.colorbar(im1, ax=ax1)
    
    # Effect size heatmap
    im2 = ax2.imshow(effect_matrix, cmap='RdBu_r', center=0)
    ax2.set_xticks(range(n_models))
    ax2.set_yticks(range(n_models))
    ax2.set_xticklabels(models, rotation=45)
    ax2.set_yticklabels(models)
    ax2.set_title(f'Effect Size\n{metric_type} - {test_type}')
    
    # Add text annotations
    for i in range(n_models):
        for j in range(n_models):
            if i != j:
                text = f'{effect_matrix[i, j]:.3f}'
                ax2.text(j, i, text, ha="center", va="center", 
                        color="white" if abs(effect_matrix[i, j]) > 0.5 else "black")
    
    plt.colorbar(im2, ax=ax2)
    plt.tight_layout()
    
    return fig

# =============================================================================
# 4. COMPREHENSIVE ANALYSIS WORKFLOW
# =============================================================================

def comprehensive_correlation_analysis(model_paths, model_names, 
                                     datasets=['test', 'train', 'val'],
                                     stratify_columns=['timepoint', 'tissue'],
                                     save_plots=False, 
                                     output_dir='correlation_analysis'):
    """
    Run a comprehensive analysis of correlations across multiple models
    
    Parameters:
    -----------
    model_paths : list
        Paths to model files
    model_names : list  
        Names for each model
    datasets : list
        Which datasets to analyze
    stratify_columns : list
        Columns to use for stratified analysis
    save_plots : bool
        Whether to save plots
    output_dir : str
        Directory to save plots
    
    Returns:
    --------
    Dictionary containing all analysis results
    """
    
    if save_plots:
        import os
        os.makedirs(output_dir, exist_ok=True)
    
    print("="*60)
    print("COMPREHENSIVE CORRELATION ANALYSIS")
    print("="*60)
    
    # 1. Load data
    print("\n1. Loading data...")
    combined_obs, combined_var = load_multiple_models(model_paths, model_names)
    
    # 2. Summary statistics
    print("\n2. Computing summary statistics...")
    summary_stats = compare_model_performance(combined_obs, combined_var, datasets)
    print("\nSummary Statistics:")
    print(summary_stats.round(4))
    
    # 3. Statistical tests
    print("\n3. Performing statistical tests...")
    test_results = perform_statistical_tests(combined_obs, combined_var, datasets)
    if len(test_results) > 0:
        print("\nStatistical Test Results:")
        significant_results = test_results[test_results['p_value'] < 0.05]
        print(f"Found {len(significant_results)} significant comparisons out of {len(test_results)} total")
        if len(significant_results) > 0:
            print(significant_results[['model1', 'model2', 'metric_type', 'test', 'p_value', 'effect_size']].round(4))
    
    # 4. Create visualizations
    print("\n4. Creating visualizations...")
    figures = {}
    
    for dataset in datasets:
        # Main correlation histograms
        fig = plot_correlation_histograms(combined_obs, combined_var, dataset=dataset)
        figures[f'correlations_{dataset}'] = fig
        if save_plots:
            fig.savefig(f'{output_dir}/correlations_{dataset}.png', dpi=300, bbox_inches='tight')
        
        # Stratified analysis
        # for col in stratify_columns:
        #     if col in combined_obs.columns:
        #         try:
        #             fig = plot_stratified_histograms(combined_obs, stratify_by=col, dataset=dataset)
        #             figures[f'stratified_{col}_{dataset}'] = fig
        #             if save_plots:
        #                 fig.savefig(f'{output_dir}/stratified_{col}_{dataset}.png', dpi=300, bbox_inches='tight')
        #         except Exception as e:
        #             print(f"Error creating stratified plot for {col}: {e}")
    
    # Statistical significance heatmaps
    # if len(test_results) > 0:
    #     for metric in ['pseudobulk', 'gene']:
    #         for test_type in ['Mann-Whitney U', "Welch's t-test"]:
    #             try:
    #                 fig = create_significance_heatmap(test_results, metric_type=metric, test_type=test_type)
    #                 if fig is not None:
    #                     figures[f'significance_{metric}_{test_type.replace(" ", "_")}'] = fig
    #                     if save_plots:
    #                         fig.savefig(f'{output_dir}/significance_{metric}_{test_type.replace(" ", "_")}.png', 
    #                                   dpi=300, bbox_inches='tight')
    #             except Exception as e:
    #                 print(f"Error creating significance heatmap for {metric} {test_type}: {e}")
    
    print(f"\n5. Analysis complete! Created {len(figures)} figures.")
    
    return {
        'combined_obs': combined_obs,
        'combined_var': combined_var,
        'summary_stats': summary_stats,
        'test_results': test_results,
        'figures': figures
    }

# =============================================================================
# USAGE EXAMPLE
# =============================================================================

# Example of how to use these functions:
"""
# Define your model paths and names
model_paths = [
    "/path/to/random_model_1.h5ad",
    "/path/to/random_model_2.h5ad", 
    "/path/to/human_borzoi_model_1.h5ad",
    "/path/to/human_borzoi_model_2.h5ad"
]

model_names = [
    "Random_1",
    "Random_2", 
    "Human_Borzoi_1",
    "Human_Borzoi_2"
]

# Run comprehensive analysis
results = comprehensive_correlation_analysis(
    model_paths=model_paths,
    model_names=model_names,
    datasets=['test', 'train', 'val'],
    stratify_columns=['timepoint', 'tissue', 'zebrafish_anatomy_ontology_class_fine'],
    save_plots=True,
    output_dir='correlation_analysis_output'
)

# Access individual results
summary_stats = results['summary_stats']
test_results = results['test_results']
figures = results['figures']

# Show specific plots
plt.show()  # Will display all generated figures
"""

print("All analysis functions loaded successfully!")
print("Use comprehensive_correlation_analysis() to run the full analysis pipeline.")

In [ ]:
# Define your model paths and names
model_paths = [
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/0/task_0/lr_1.154578199918017e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-0_20250529_Random_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/5/task_5/lr_7.116198906552475e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-5_20250529_Random_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/1/task_1/lr_1.534060288605152e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-1_20250529_Random_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/2/task_2/lr_2.9680669393835383e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-2_20250529_Random_3.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/3/task_3/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-3_20250529_Human_Borzoi_3.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/2/task_2/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-2_20250529_Human_Borzoi_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/1/task_1/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-1_20250529_Human_Borzoi_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-0_20250529_Human_Borzoi_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-0_20250616_Mouse_Borzoi_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/1/task_1/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-1_20250616_Mouse_Borzoi_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/2/task_2/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-2_20250616_Mouse_Borzoi_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/3/task_3/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-3_20250616_Mouse_Borzoi_3.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20473753/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20473753-0_20250616_Human_Decima_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491529/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491529-0_20250616_Human_Decima_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491962/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491962-0_20250616_Human_Decima_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491359/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491359-0_20250616_Human_Decima_3.h5ad"
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed42/version_0/data_out_decima_random_lr3e-06_seed42_20250619_Random_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed43/version_0/data_out_decima_random_lr3e-06_seed43_20250619_Random_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed44/version_0/data_out_decima_random_lr3e-06_seed44_20250619_Random_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed45/version_0/data_out_decima_random_lr3e-06_seed45_20250619_Random_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep0_lr3e-05_seed42_20250619_Human_Borzoi_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep1_lr3e-05_seed42_20250619_Human_Borzoi_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep2_lr3e-05_seed42_20250619_Human_Borzoi_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep3_lr3e-05_seed42_20250619_Human_Borzoi_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep0_lr3e-05_seed42_20250619_Mouse_Borzoi_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep1_lr3e-05_seed42_20250619_Mouse_Borzoi_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep2_lr3e-05_seed42_20250619_Mouse_Borzoi_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep3_lr3e-05_seed42_20250619_Mouse_Borzoi_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep0_lr3e-05_seed42_20250619_Human_Decima_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep1_lr3e-05_seed42_20250619_Human_Decima_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep2_lr3e-05_seed42_20250619_Human_Decima_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep3_lr3e-05_seed42_20250619_Human_Decima_3.h5ad"
]

model_names = [
    "Random_0",
    "Random_1",
    "Random_2", 
    "Random_3",
    "Human_Borzoi_0",
    "Human_Borzoi_1",
    "Human_Borzoi_2",
    "Human_Borzoi_3",
    "Mouse_Borzoi_0",
    "Mouse_Borzoi_1",
    "Mouse_Borzoi_2",
    "Mouse_Borzoi_3",
    "Human_Decima_0",
    "Human_Decima_1",
    "Human_Decima_2",
    "Human_Decima_3"
]

# Run comprehensive analysis
results = comprehensive_correlation_analysis(
    model_paths=model_paths,
    model_names=model_names,
    datasets=['test', 'train', 'val'],
    stratify_columns=['timepoint', 'tissue', 'zebrafish_anatomy_ontology_class_fine'],
    save_plots=True,
    output_dir='correlation_analysis_output'
)

# Access individual results
summary_stats = results['summary_stats']
test_results = results['test_results']
figures = results['figures']

# Show specific plots
plt.show()  # Will display all generated figures

### Figure 3 in DanioDecima MS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

def create_clear_model_comparison_improved(combined_obs, combined_var, dataset='test', figsize=(24, 14)):
    """
    Create clear, non-overlapping visualizations comparing model performance with improved layout
    """
    
    # Set up the figure with subplots - increased spacing
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(3, 4, height_ratios=[2.5, 2.5, 1.2], hspace=0.4, wspace=0.35)
    
    # Define consistent colors for each model group
    group_colors = {
        'Random': '#1f77b4',      # Blue
        'Mouse_Borzoi': '#ff7f0e', # Orange  
        'Human_Borzoi': '#2ca02c', # Green
        'Human_Decima': '#d62728'  # Red
    }
    
    # Extract model group from model_type
    def get_model_group(model_type):
        if 'Random' in model_type:
            return 'Random'
        elif 'Human_Borzoi' in model_type:
            return 'Human_Borzoi'
        elif 'Mouse_Borzoi' in model_type:
            return 'Mouse_Borzoi'
        elif 'Human_Decima' in model_type:
            return 'Human_Decima'
        else:
            return 'Other'
    
    combined_obs['model_group'] = combined_obs['model_type'].apply(get_model_group)
    combined_var['model_group'] = combined_var['model_type'].apply(get_model_group)
    
    # Reorder groups as requested: Random, Mouse_Borzoi, Human_Borzoi, Human_Decima
    group_order = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # 1. PSEUDOBULK CORRELATIONS - Separate histograms
    col_name = f"{dataset}_pearson"
    if col_name in combined_obs.columns:
        
        # Individual histograms for each group
        for i, group in enumerate(group_order):
            ax = fig.add_subplot(gs[0, i])
            
            group_data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
            
            if len(group_data) > 0:
                # Create histogram
                ax.hist(group_data, bins=25, alpha=0.7, color=group_colors[group], 
                       density=True, edgecolor='black', linewidth=0.5)
                
                # Add statistics text
                mean_val = group_data.mean()
                std_val = group_data.std()
                median_val = group_data.median()
                
                ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
                ax.axvline(median_val, color='orange', linestyle=':', linewidth=2)
                
                # Clean title without overlapping text
                ax.set_title(f'{group.replace("_", " ")}\nPseudobulk Correlations (n={len(group_data)})', 
                           fontweight='bold', fontsize=12, pad=15)
                ax.set_xlabel('Pearson Correlation', fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                
                # Add legend with better positioning
                ax.legend([f'Mean: {mean_val:.3f}', f'Median: {median_val:.3f}'], 
                         loc='upper left', fontsize=9, framealpha=0.9)
                ax.grid(True, alpha=0.3)
                
                # Set consistent x-axis limits
                ax.set_xlim(0.4, 0.85)
                # Improve tick labels
                ax.tick_params(axis='both', which='major', labelsize=9)
    
    # 2. GENE CORRELATIONS - Separate histograms  
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    
    if len(dataset_var) > 0:
        for i, group in enumerate(group_order):
            ax = fig.add_subplot(gs[1, i])
            
            group_data = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
            
            if len(group_data) > 0:
                # Create histogram
                ax.hist(group_data, bins=25, alpha=0.7, color=group_colors[group],
                       density=True, edgecolor='black', linewidth=0.5)
                
                # Add statistics
                mean_val = group_data.mean()
                std_val = group_data.std()
                median_val = group_data.median()
                
                ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
                ax.axvline(median_val, color='orange', linestyle=':', linewidth=2)
                
                # Clean title
                ax.set_title(f'{group.replace("_", " ")}\nGene Correlations (n={len(group_data)})', 
                           fontweight='bold', fontsize=12, pad=15)
                ax.set_xlabel('Pearson Correlation', fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                
                # Add legend
                ax.legend([f'Mean: {mean_val:.3f}', f'Median: {median_val:.3f}'], 
                         loc='upper left', fontsize=9, framealpha=0.9)
                ax.grid(True, alpha=0.3)
                
                # Extended y-axis range for gene correlations as requested
                ax.set_xlim(-0.75, 1.0)
                ax.tick_params(axis='both', which='major', labelsize=9)
    
    # 3. COMPARATIVE BOX PLOTS - Improved layout
    ax_box = fig.add_subplot(gs[2, :])
    
    # Prepare data for box plots in the correct order
    pseudobulk_data = []
    gene_data = []
    
    for group in group_order:
        # Pseudobulk data
        pb_data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
        if len(pb_data) > 0:
            pseudobulk_data.append(pb_data)
        
        # Gene data
        gene_data_group = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
        if len(gene_data_group) > 0:
            gene_data.append(gene_data_group)
    
    # Create side-by-side box plots
    all_data = pseudobulk_data + gene_data
    
    # Improved labels - shorter and cleaner
    pseudobulk_labels = [group.replace('_', '\n') for group in group_order]
    gene_labels = [group.replace('_', '\n') for group in group_order]
    all_labels = pseudobulk_labels + gene_labels
    
    if all_data:
        bp = ax_box.boxplot(all_data, labels=all_labels, patch_artist=True, showfliers=False)
        
        # Color the boxes in the correct order
        colors_extended = [group_colors[group] for group in group_order] * 2
        for patch, color in zip(bp['boxes'], colors_extended):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax_box.set_title('Comparative Performance: Pseudobulk vs Gene Correlations', 
                        fontweight='bold', fontsize=14, pad=20)
        ax_box.set_ylabel('Pearson Correlation', fontsize=12)
        ax_box.grid(True, alpha=0.3)
        
        # Improve tick label formatting - no rotation needed with shorter labels
        ax_box.tick_params(axis='x', labelsize=10)
        ax_box.tick_params(axis='y', labelsize=10)
        
        # Add vertical line to separate pseudobulk from gene correlations
        ax_box.axvline(4.5, color='black', linestyle='-', alpha=0.5, linewidth=2)
        
        # Add section labels with better positioning
        y_pos = ax_box.get_ylim()[1] * 0.95
        ax_box.text(2.5, y_pos, 'Pseudobulk', ha='center', fontweight='bold', fontsize=12,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.7))
        ax_box.text(6.5, y_pos, 'Genes', ha='center', fontweight='bold', fontsize=12,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.7))
    
    # Main title with better spacing
    plt.suptitle(f'Model Performance Comparison - {dataset.upper()} Dataset', 
                 fontsize=18, fontweight='bold', y=0.96)
    
    # Adjust layout to prevent overlapping
    plt.tight_layout(rect=[0, 0.03, 1, 0.94])
    
    return fig

def perform_pairwise_statistical_tests_ordered(combined_obs, combined_var, dataset='test'):
    """
    Perform comprehensive pairwise statistical comparisons with correct ordering
    """
    
    def get_model_group(model_type):
        if 'Random' in model_type:
            return 'Random'
        elif 'Human_Borzoi' in model_type:
            return 'Human_Borzoi'
        elif 'Mouse_Borzoi' in model_type:
            return 'Mouse_Borzoi'
        elif 'Human_Decima' in model_type:
            return 'Human_Decima'
        else:
            return 'Other'
    
    combined_obs['model_group'] = combined_obs['model_type'].apply(get_model_group)
    combined_var['model_group'] = combined_var['model_type'].apply(get_model_group)
    
    results = []
    # Use the correct order
    groups = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # Pseudobulk comparisons
    col_name = f"{dataset}_pearson"
    if col_name in combined_obs.columns:
        
        # Prepare data
        group_data = {}
        for group in groups:
            data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
            if len(data) > 0:
                group_data[group] = data
        
        # Pairwise comparisons
        for i, group1 in enumerate(groups):
            for j, group2 in enumerate(groups):
                if i < j and group1 in group_data and group2 in group_data:
                    data1, data2 = group_data[group1], group_data[group2]
                    
                    # Mann-Whitney U test
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = (data1.median() - data2.median()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'metric_type': 'Pseudobulk',
                            'group1': group1,
                            'group2': group2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'median_diff': data1.median() - data2.median(),
                            'group1_mean': data1.mean(),
                            'group2_mean': data2.mean(),
                            'group1_n': len(data1),
                            'group2_n': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in pseudobulk comparison {group1} vs {group2}: {e}")
    
    # Gene comparisons
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    if len(dataset_var) > 0:
        
        # Prepare data
        group_data = {}
        for group in groups:
            data = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
            if len(data) > 0:
                group_data[group] = data
        
        # Pairwise comparisons
        for i, group1 in enumerate(groups):
            for j, group2 in enumerate(groups):
                if i < j and group1 in group_data and group2 in group_data:
                    data1, data2 = group_data[group1], group_data[group2]
                    
                    # Mann-Whitney U test
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = (data1.median() - data2.median()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'metric_type': 'Gene',
                            'group1': group1,
                            'group2': group2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'median_diff': data1.median() - data2.median(),
                            'group1_mean': data1.mean(),
                            'group2_mean': data2.mean(),
                            'group1_n': len(data1),
                            'group2_n': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in gene comparison {group1} vs {group2}: {e}")
    
    # Convert to DataFrame and apply multiple testing correction
    results_df = pd.DataFrame(results)
    
    if len(results_df) > 0:
        # Apply FDR correction within each metric type
        for metric_type in results_df['metric_type'].unique():
            mask = results_df['metric_type'] == metric_type
            if mask.sum() > 1:
                _, p_corrected, _, _ = multipletests(
                    results_df.loc[mask, 'p_value'], 
                    method='fdr_bh'
                )
                results_df.loc[mask, 'p_corrected'] = p_corrected
            else:
                results_df.loc[mask, 'p_corrected'] = results_df.loc[mask, 'p_value']
    
    return results_df

def create_statistical_summary_table_ordered(test_results):
    """
    Create a nicely formatted summary table with correct ordering
    """
    if len(test_results) == 0:
        print("No statistical test results to display")
        return
    
    print("\n" + "="*120)
    print("STATISTICAL COMPARISON RESULTS (Ordered: Random → Mouse Borzoi → Human Borzoi → Human Decima)")
    print("="*120)
    
    # Define the order for display
    group_order = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    for metric_type in ['Pseudobulk', 'Gene']:
        metric_results = test_results[test_results['metric_type'] == metric_type]
        
        if len(metric_results) > 0:
            print(f"\n{metric_type.upper()} CORRELATIONS:")
            print("-" * 60)
            
            # Sort results by the order we want
            def get_order_score(row):
                g1_idx = group_order.index(row['group1']) if row['group1'] in group_order else 999
                g2_idx = group_order.index(row['group2']) if row['group2'] in group_order else 999
                return (g1_idx, g2_idx)
            
            metric_results_sorted = metric_results.copy()
            metric_results_sorted['order_score'] = metric_results_sorted.apply(get_order_score, axis=1)
            metric_results_sorted = metric_results_sorted.sort_values('order_score')
            
            for _, row in metric_results_sorted.iterrows():
                significance = ""
                p_val = row['p_corrected'] if 'p_corrected' in row else row['p_value']
                
                if p_val < 0.001:
                    significance = "***"
                elif p_val < 0.01:
                    significance = "**"
                elif p_val < 0.05:
                    significance = "*"
                else:
                    significance = "ns"
                
                # Clean up group names for display
                group1_clean = row['group1'].replace('_', ' ')
                group2_clean = row['group2'].replace('_', ' ')
                
                print(f"{group1_clean} vs {group2_clean}:")
                print(f"  Mean difference: {row['mean_diff']:+.4f}")
                print(f"  Effect size: {row['effect_size']:+.4f}")
                print(f"  P-value: {p_val:.4f} {significance}")
                print(f"  Sample sizes: {row['group1_n']} vs {row['group2_n']}")
                print()

def create_performance_ranking_table_ordered(combined_obs, combined_var, dataset='test'):
    """
    Create a ranking table with correct ordering
    """
    
    def get_model_group(model_type):
        if 'Random' in model_type:
            return 'Random'
        elif 'Human_Borzoi' in model_type:
            return 'Human_Borzoi'
        elif 'Mouse_Borzoi' in model_type:
            return 'Mouse_Borzoi'
        elif 'Human_Decima' in model_type:
            return 'Human_Decima'
        else:
            return 'Other'
    
    combined_obs['model_group'] = combined_obs['model_type'].apply(get_model_group)
    combined_var['model_group'] = combined_var['model_type'].apply(get_model_group)
    
    ranking_data = []
    groups = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # Pseudobulk rankings
    col_name = f"{dataset}_pearson"
    if col_name in combined_obs.columns:
        for group in groups:
            data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
            if len(data) > 0:
                ranking_data.append({
                    'Model_Group': group.replace('_', ' '),
                    'Metric': 'Pseudobulk',
                    'Mean': data.mean(),
                    'Median': data.median(),
                    'Std': data.std(),
                    'N': len(data),
                    'Q25': data.quantile(0.25),
                    'Q75': data.quantile(0.75)
                })
    
    # Gene rankings
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    if len(dataset_var) > 0:
        for group in groups:
            data = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
            if len(data) > 0:
                ranking_data.append({
                    'Model_Group': group.replace('_', ' '),
                    'Metric': 'Gene',
                    'Mean': data.mean(),
                    'Median': data.median(),
                    'Std': data.std(),
                    'N': len(data),
                    'Q25': data.quantile(0.25),
                    'Q75': data.quantile(0.75)
                })
    
    ranking_df = pd.DataFrame(ranking_data)
    
    print(f"\n🏆 MODEL PERFORMANCE RANKING - {dataset.upper()} DATASET")
    print("="*80)
    
    for metric in ['Pseudobulk', 'Gene']:
        metric_data = ranking_df[ranking_df['Metric'] == metric].copy()
        if len(metric_data) > 0:
            metric_data = metric_data.sort_values('Mean', ascending=False)
            
            print(f"\n{metric.upper()} CORRELATIONS (ranked by mean):")
            print("-" * 60)
            
            for i, (_, row) in enumerate(metric_data.iterrows(), 1):
                emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📍"
                print(f"{emoji} {i}. {row['Model_Group']}")
                print(f"     Mean: {row['Mean']:.4f} ± {row['Std']:.4f}")
                print(f"     Median: {row['Median']:.4f}")
                print(f"     IQR: [{row['Q25']:.4f}, {row['Q75']:.4f}]")
                print(f"     N: {row['N']}")
                print()
    
    return ranking_df

# Usage with your existing data:
print("\n" + "="*60)
print("CREATING IMPROVED MODEL COMPARISON")
print("="*60)

# 0. Load datasets
combined_obs, combined_var = load_multiple_models(model_paths, model_names)

# 1. Create the improved comparison plot
fig = create_clear_model_comparison_improved(combined_obs, combined_var, dataset='test')
plt.show()

# 2. Perform statistical tests with correct ordering
print("\nPerforming statistical tests...")
test_results = perform_pairwise_statistical_tests_ordered(combined_obs, combined_var, dataset='test')

# 3. Display statistical summary with correct ordering
create_statistical_summary_table_ordered(test_results)

# 4. Show performance rankings
ranking_df = create_performance_ranking_table_ordered(combined_obs, combined_var, dataset='test')

print("\nImproved analysis complete!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

def add_significance_brackets(ax, data_groups, group_positions, test_results, metric_type, y_offset_start=0.05):
    """
    Add significance brackets and symbols above boxplots
    """
    # Get the y-axis limits to position brackets appropriately
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    
    # Filter test results for this metric type
    metric_results = test_results[test_results['metric_type'] == metric_type]
    
    # Group order for positioning
    group_order = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # Create position mapping
    pos_map = {}
    for i, group in enumerate(group_order):
        pos_map[group] = group_positions[i]
    
    # Sort comparisons by distance between groups to avoid overlapping brackets
    comparisons = []
    for _, row in metric_results.iterrows():
        group1, group2 = row['group1'], row['group2']
        if group1 in pos_map and group2 in pos_map:
            pos1, pos2 = pos_map[group1], pos_map[group2]
            distance = abs(pos2 - pos1)
            p_val = row['p_corrected'] if 'p_corrected' in row else row['p_value']
            
            # Only show significant results
            if p_val < 0.05:
                comparisons.append({
                    'group1': group1,
                    'group2': group2,
                    'pos1': pos1,
                    'pos2': pos2,
                    'distance': distance,
                    'p_value': p_val,
                    'effect_size': row['effect_size']
                })
    
    # Sort by distance (shorter distances first, then longer ones on top)
    comparisons.sort(key=lambda x: x['distance'])
    
    # Add brackets
    bracket_height = y_range * 0.03  # Height of each bracket level
    current_level = 0
    
    for i, comp in enumerate(comparisons):
        # Determine significance symbol
        p_val = comp['p_value']
        if p_val < 0.001:
            sig_symbol = "***"
        elif p_val < 0.01:
            sig_symbol = "**"
        elif p_val < 0.05:
            sig_symbol = "*"
        else:
            continue  # Skip non-significant
        
        # Calculate bracket position
        x1, x2 = comp['pos1'], comp['pos2']
        y_bracket = y_max + y_range * (y_offset_start + current_level * 0.08)
        
        # Draw bracket
        ax.plot([x1, x1, x2, x2], 
                [y_bracket - bracket_height/2, y_bracket, y_bracket, y_bracket - bracket_height/2], 
                'k-', linewidth=1)
        
        # Add significance symbol
        ax.text((x1 + x2) / 2, y_bracket + bracket_height/2, sig_symbol, 
                ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        current_level += 1
    
    # Adjust y-axis to accommodate brackets
    if comparisons:
        new_y_max = y_max + y_range * (y_offset_start + len(comparisons) * 0.08 + 0.05)
        ax.set_ylim(y_min, new_y_max)

def create_enhanced_model_comparison(combined_obs, combined_var, dataset='test', figsize=(24, 16)):
    """
    Create enhanced model comparison with larger boxplots and statistical annotations
    """
    
    # Set up the figure with subplots - more space for boxplots
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(4, 4, height_ratios=[2, 2, 1.5, 1.5], hspace=0.4, wspace=0.35)
    
    # Define consistent colors for each model group
    group_colors = {
        'Random': '#1f77b4',      # Blue
        'Mouse_Borzoi': '#ff7f0e', # Orange  
        'Human_Borzoi': '#2ca02c', # Green
        'Human_Decima': '#d62728'  # Red
    }
    
    # Extract model group from model_type
    def get_model_group(model_type):
        if 'Random' in model_type:
            return 'Random'
        elif 'Human_Borzoi' in model_type:
            return 'Human_Borzoi'
        elif 'Mouse_Borzoi' in model_type:
            return 'Mouse_Borzoi'
        elif 'Human_Decima' in model_type:
            return 'Human_Decima'
        else:
            return 'Other'
    
    combined_obs['model_group'] = combined_obs['model_type'].apply(get_model_group)
    combined_var['model_group'] = combined_var['model_type'].apply(get_model_group)
    
    # Reorder groups as requested: Random, Mouse_Borzoi, Human_Borzoi, Human_Decima
    group_order = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # 1. PSEUDOBULK CORRELATIONS - Separate histograms
    col_name = f"{dataset}_pearson"
    if col_name in combined_obs.columns:
        
        # Individual histograms for each group
        for i, group in enumerate(group_order):
            ax = fig.add_subplot(gs[0, i])
            
            group_data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
            
            if len(group_data) > 0:
                # Create histogram
                ax.hist(group_data, bins=25, alpha=0.7, color=group_colors[group], 
                       density=True, edgecolor='black', linewidth=0.5)
                
                # Add statistics text
                mean_val = group_data.mean()
                std_val = group_data.std()
                median_val = group_data.median()
                
                ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
                ax.axvline(median_val, color='orange', linestyle=':', linewidth=2)
                
                # Clean title without overlapping text
                ax.set_title(f'{group.replace("_", " ")}\nPseudobulk Correlations (n={len(group_data)})', 
                           fontweight='bold', fontsize=12, pad=15)
                ax.set_xlabel('Pearson Correlation', fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                
                # Add legend with better positioning
                ax.legend([f'Mean: {mean_val:.3f}', f'Median: {median_val:.3f}'], 
                         loc='upper left', fontsize=9, framealpha=0.9)
                ax.grid(True, alpha=0.3)
                
                # Set consistent x-axis limits
                ax.set_xlim(0.4, 0.85)
                # Improve tick labels
                ax.tick_params(axis='both', which='major', labelsize=9)
    
    # 2. GENE CORRELATIONS - Separate histograms  
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    
    if len(dataset_var) > 0:
        for i, group in enumerate(group_order):
            ax = fig.add_subplot(gs[1, i])
            
            group_data = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
            
            if len(group_data) > 0:
                # Create histogram
                ax.hist(group_data, bins=25, alpha=0.7, color=group_colors[group],
                       density=True, edgecolor='black', linewidth=0.5)
                
                # Add statistics
                mean_val = group_data.mean()
                std_val = group_data.std()
                median_val = group_data.median()
                
                ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
                ax.axvline(median_val, color='orange', linestyle=':', linewidth=2)
                
                # Clean title
                ax.set_title(f'{group.replace("_", " ")}\nGene Correlations (n={len(group_data)})', 
                           fontweight='bold', fontsize=12, pad=15)
                ax.set_xlabel('Pearson Correlation', fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                
                # Add legend
                ax.legend([f'Mean: {mean_val:.3f}', f'Median: {median_val:.3f}'], 
                         loc='upper left', fontsize=9, framealpha=0.9)
                ax.grid(True, alpha=0.3)
                
                # Extended y-axis range for gene correlations as requested
                ax.set_xlim(-0.75, 1.0)
                ax.tick_params(axis='both', which='major', labelsize=9)
    
    # Calculate statistical tests for annotations
    test_results = perform_pairwise_statistical_tests_ordered(combined_obs, combined_var, dataset)
    
    # 3. PSEUDOBULK BOX PLOTS - Enhanced with statistical annotations
    ax_pb = fig.add_subplot(gs[2, :])
    
    # Prepare pseudobulk data
    pseudobulk_data = []
    for group in group_order:
        pb_data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
        if len(pb_data) > 0:
            pseudobulk_data.append(pb_data)
    
    if pseudobulk_data:
        # Create boxplot
        pb_labels = [group.replace('_', '\n') for group in group_order]
        bp_pb = ax_pb.boxplot(pseudobulk_data, labels=pb_labels, patch_artist=True, showfliers=False)
        
        # Color the boxes
        for patch, group in zip(bp_pb['boxes'], group_order):
            patch.set_facecolor(group_colors[group])
            patch.set_alpha(0.7)
        
        # Add significance brackets
        group_positions = list(range(1, len(group_order) + 1))
        add_significance_brackets(ax_pb, pseudobulk_data, group_positions, test_results, 'Pseudobulk')
        
        ax_pb.set_title('Pseudobulk Correlations with Statistical Comparisons', 
                       fontweight='bold', fontsize=14, pad=20)
        ax_pb.set_ylabel('Pearson Correlation', fontsize=12)
        ax_pb.grid(True, alpha=0.3)
        ax_pb.tick_params(axis='both', labelsize=11)
    
    # 4. GENE BOX PLOTS - Enhanced with statistical annotations
    ax_gene = fig.add_subplot(gs[3, :])
    
    # Prepare gene data
    gene_data = []
    for group in group_order:
        gene_data_group = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
        if len(gene_data_group) > 0:
            gene_data.append(gene_data_group)
    
    if gene_data:
        # Create boxplot
        gene_labels = [group.replace('_', '\n') for group in group_order]
        bp_gene = ax_gene.boxplot(gene_data, labels=gene_labels, patch_artist=True, showfliers=False)
        
        # Color the boxes
        for patch, group in zip(bp_gene['boxes'], group_order):
            patch.set_facecolor(group_colors[group])
            patch.set_alpha(0.7)
        
        # Add significance brackets
        group_positions = list(range(1, len(group_order) + 1))
        add_significance_brackets(ax_gene, gene_data, group_positions, test_results, 'Gene')
        
        ax_gene.set_title('Gene Correlations with Statistical Comparisons', 
                         fontweight='bold', fontsize=14, pad=20)
        ax_gene.set_ylabel('Pearson Correlation', fontsize=12)
        ax_gene.grid(True, alpha=0.3)
        ax_gene.tick_params(axis='both', labelsize=11)
    
    # Main title with better spacing
    plt.suptitle(f'Enhanced Model Performance Comparison - {dataset.upper()} Dataset', 
                 fontsize=18, fontweight='bold', y=0.96)
    
    # Add legend for significance symbols
    legend_text = "Statistical significance: * p<0.05, ** p<0.01, *** p<0.001"
    fig.text(0.5, 0.02, legend_text, ha='center', fontsize=12, style='italic')
    
    # Adjust layout to prevent overlapping
    plt.tight_layout(rect=[0, 0.05, 1, 0.94])
    
    return fig

def perform_pairwise_statistical_tests_ordered(combined_obs, combined_var, dataset='test'):
    """
    Perform comprehensive pairwise statistical comparisons with correct ordering
    """
    
    def get_model_group(model_type):
        if 'Random' in model_type:
            return 'Random'
        elif 'Human_Borzoi' in model_type:
            return 'Human_Borzoi'
        elif 'Mouse_Borzoi' in model_type:
            return 'Mouse_Borzoi'
        elif 'Human_Decima' in model_type:
            return 'Human_Decima'
        else:
            return 'Other'
    
    combined_obs['model_group'] = combined_obs['model_type'].apply(get_model_group)
    combined_var['model_group'] = combined_var['model_type'].apply(get_model_group)
    
    results = []
    # Use the correct order
    groups = ['Random', 'Mouse_Borzoi', 'Human_Borzoi', 'Human_Decima']
    
    # Pseudobulk comparisons
    col_name = f"{dataset}_pearson"
    if col_name in combined_obs.columns:
        
        # Prepare data
        group_data = {}
        for group in groups:
            data = combined_obs[combined_obs['model_group'] == group][col_name].dropna()
            if len(data) > 0:
                group_data[group] = data
        
        # Pairwise comparisons
        for i, group1 in enumerate(groups):
            for j, group2 in enumerate(groups):
                if i < j and group1 in group_data and group2 in group_data:
                    data1, data2 = group_data[group1], group_data[group2]
                    
                    # Mann-Whitney U test
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = (data1.median() - data2.median()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'metric_type': 'Pseudobulk',
                            'group1': group1,
                            'group2': group2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'median_diff': data1.median() - data2.median(),
                            'group1_mean': data1.mean(),
                            'group2_mean': data2.mean(),
                            'group1_n': len(data1),
                            'group2_n': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in pseudobulk comparison {group1} vs {group2}: {e}")
    
    # Gene comparisons
    dataset_var = combined_var[combined_var['dataset'] == dataset]
    if len(dataset_var) > 0:
        
        # Prepare data
        group_data = {}
        for group in groups:
            data = dataset_var[dataset_var['model_group'] == group]['pearson'].dropna()
            if len(data) > 0:
                group_data[group] = data
        
        # Pairwise comparisons
        for i, group1 in enumerate(groups):
            for j, group2 in enumerate(groups):
                if i < j and group1 in group_data and group2 in group_data:
                    data1, data2 = group_data[group1], group_data[group2]
                    
                    # Mann-Whitney U test
                    try:
                        statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        effect_size = (data1.median() - data2.median()) / np.sqrt((data1.var() + data2.var()) / 2)
                        
                        results.append({
                            'metric_type': 'Gene',
                            'group1': group1,
                            'group2': group2,
                            'test': 'Mann-Whitney U',
                            'statistic': statistic,
                            'p_value': p_value,
                            'effect_size': effect_size,
                            'mean_diff': data1.mean() - data2.mean(),
                            'median_diff': data1.median() - data2.median(),
                            'group1_mean': data1.mean(),
                            'group2_mean': data2.mean(),
                            'group1_n': len(data1),
                            'group2_n': len(data2)
                        })
                    except Exception as e:
                        print(f"Error in gene comparison {group1} vs {group2}: {e}")
    
    # Convert to DataFrame and apply multiple testing correction
    results_df = pd.DataFrame(results)
    
    if len(results_df) > 0:
        # Apply FDR correction within each metric type
        for metric_type in results_df['metric_type'].unique():
            mask = results_df['metric_type'] == metric_type
            if mask.sum() > 1:
                _, p_corrected, _, _ = multipletests(
                    results_df.loc[mask, 'p_value'], 
                    method='fdr_bh'
                )
                results_df.loc[mask, 'p_corrected'] = p_corrected
            else:
                results_df.loc[mask, 'p_corrected'] = results_df.loc[mask, 'p_value']
    
    return results_df

# Usage with your existing data:
print("\n" + "="*60)
print("CREATING ENHANCED MODEL COMPARISON WITH STATISTICAL ANNOTATIONS")
print("="*60)

# Create the enhanced comparison plot with statistical annotations
fig = create_enhanced_model_comparison(combined_obs, combined_var, dataset='test')
plt.show()

print("\nEnhanced analysis with statistical annotations complete!")

### Diagnostics of read counts vs Pearson correlation

In [ ]:
def plot_total_counts_vs_test_pearson(ad, 
                                    model_name="Model", 
                                    figsize=(12, 8),
                                    color_by=None,
                                    log_scale=True,
                                    add_correlation_stats=True,
                                    save_path=None,
                                    show_poorest_performers=True,
                                    n_poorest=5):
    """
    Create a comprehensive plot of total counts vs test Pearson correlation for pseudobulks
    
    Parameters:
    -----------
    ad : AnnData object
        The pseudobulk data
    model_name : str
        Name of the model for the title
    figsize : tuple
        Figure size
    color_by : str, optional
        Column name to color points by (e.g., 'cell_type', 'tissue', 'organ')
    log_scale : bool
        Whether to use log scale for total_counts
    add_correlation_stats : bool
        Whether to add correlation statistics to the plot
    save_path : str, optional
        Path to save the figure
    show_poorest_performers : bool
        Whether to identify and display poorest performing pseudobulks
    n_poorest : int
        Number of poorest performers to display
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    from scipy import stats
    import pandas as pd
    
    # Create the plot
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle(f'{model_name}: Total Counts vs Test Pearson Correlation Analysis', 
                 fontsize=16, fontweight='bold')
    
    # Prepare data
    df = ad.obs.copy()
    df = df.dropna(subset=['total_counts', 'test_pearson'])
    
    # Calculate correlation statistics
    if add_correlation_stats:
        pearson_r, pearson_p = stats.pearsonr(df['total_counts'], df['test_pearson'])
        spearman_r, spearman_p = stats.spearmanr(df['total_counts'], df['test_pearson'])
    
    # Identify poorest performers
    if show_poorest_performers:
        poorest_performers = df.nsmallest(n_poorest, 'test_pearson')
        # Highlight them in the scatter plot
        poorest_indices = poorest_performers.index
    
    # Plot 1: Scatter plot with regression line
    ax1 = axes[0, 0]
    if color_by and color_by in df.columns:
        unique_categories = df[color_by].unique()
        colors = plt.cm.Set1(np.linspace(0, 1, len(unique_categories)))
        for i, category in enumerate(unique_categories):
            mask = df[color_by] == category
            ax1.scatter(df[mask]['total_counts'], df[mask]['test_pearson'], 
                       alpha=0.6, s=30, c=[colors[i]], label=category)
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    else:
        ax1.scatter(df['total_counts'], df['test_pearson'], alpha=0.6, s=30, color='blue')
    
    # Highlight poorest performers
    if show_poorest_performers:
        ax1.scatter(poorest_performers['total_counts'], poorest_performers['test_pearson'], 
                   color='red', s=80, alpha=0.8, marker='x', linewidth=3, 
                   label=f'Poorest {n_poorest} performers')
        ax1.legend()
    
    # # Add regression line
    # z = np.polyfit(df['total_counts'], df['test_pearson'], 1)
    # p = np.poly1d(z)
    # ax1.plot(df['total_counts'], p(df['total_counts']), "r--", alpha=0.8)
    
    if log_scale:
        ax1.set_xscale('log')
    ax1.set_xlabel('Total Counts (log scale)' if log_scale else 'Total Counts')
    ax1.set_ylabel('Test Pearson Correlation')
    ax1.set_title('Scatter Plot')
    ax1.grid(True, alpha=0.3)
    
    if add_correlation_stats:
        ax1.text(0.05, 0.95, f'Pearson r: {pearson_r:.3f} (p={pearson_p:.2e})\nSpearman ρ: {spearman_r:.3f} (p={spearman_p:.2e})', 
                transform=ax1.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 2: Hexbin plot for density
    ax2 = axes[0, 1]
    if log_scale:
        hb = ax2.hexbin(np.log10(df['total_counts']), df['test_pearson'], 
                       gridsize=30, cmap='Blues', mincnt=1)
        ax2.set_xlabel('Log10(Total Counts)')
    else:
        hb = ax2.hexbin(df['total_counts'], df['test_pearson'], 
                       gridsize=30, cmap='Blues', mincnt=1)
        ax2.set_xlabel('Total Counts')
    
    ax2.set_ylabel('Test Pearson Correlation')
    ax2.set_title('Density Plot (Hexbin)')
    plt.colorbar(hb, ax=ax2, label='Count')
    
    # Plot 3: Distribution of Total Counts
    ax3 = axes[1, 0]
    ax3.hist(df['total_counts'], bins=50, alpha=0.7, edgecolor='black')
    if log_scale:
        ax3.set_xscale('log')
    ax3.set_xlabel('Total Counts (log scale)' if log_scale else 'Total Counts')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Distribution of Total Counts')
    ax3.grid(True, alpha=0.3)
    
    # Add statistics
    ax3.text(0.05, 0.95, f'N pseudobulks: {len(df)}\nMean: {df["total_counts"].mean():.0f}\nMedian: {df["total_counts"].median():.0f}', 
             transform=ax3.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 4: Distribution of Test Pearson
    ax4 = axes[1, 1]
    ax4.hist(df['test_pearson'], bins=50, alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Test Pearson Correlation')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Distribution of Test Pearson Correlations')
    ax4.grid(True, alpha=0.3)
    
    # Add statistics
    ax4.text(0.05, 0.95, f'Mean: {df["test_pearson"].mean():.3f}\nMedian: {df["test_pearson"].median():.3f}\nStd: {df["test_pearson"].std():.3f}', 
             transform=ax4.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print(f"SUMMARY STATISTICS FOR {model_name.upper()}")
    print("="*60)
    print(f"Number of pseudobulks: {len(df)}")
    print(f"\nTotal Counts:")
    print(f"  Mean: {df['total_counts'].mean():.0f}")
    print(f"  Median: {df['total_counts'].median():.0f}")
    print(f"  Range: {df['total_counts'].min():.0f} - {df['total_counts'].max():.0f}")
    print(f"\nTest Pearson Correlation:")
    print(f"  Mean: {df['test_pearson'].mean():.3f}")
    print(f"  Median: {df['test_pearson'].median():.3f}")
    print(f"  Range: {df['test_pearson'].min():.3f} - {df['test_pearson'].max():.3f}")
    print(f"  Std: {df['test_pearson'].std():.3f}")
    
    if add_correlation_stats:
        print(f"\nCorrelation Analysis:")
        print(f"  Pearson correlation: {pearson_r:.3f} (p-value: {pearson_p:.2e})")
        print(f"  Spearman correlation: {spearman_r:.3f} (p-value: {spearman_p:.2e})")
    
    # Display poorest performers
    if show_poorest_performers:
        print(f"\n" + "="*80)
        print(f"POOREST {n_poorest} PERFORMING PSEUDOBULKS")
        print("="*80)
        
        # Get all available metadata columns
        metadata_cols = [col for col in df.columns if col not in ['test_pearson', 'train_pearson', 'val_pearson']]
        
        # Display the poorest performers with all their metadata
        for i, (idx, row) in enumerate(poorest_performers.iterrows(), 1):
            print(f"\n{i}. PSEUDOBULK INDEX: {idx}")
            print(f"   Test Pearson: {row['test_pearson']:.4f}")
            print(f"   Total Counts: {row['total_counts']:,.0f}")
            
            # Print other metadata
            for col in metadata_cols:
                if col in ['total_counts']:  # Skip already printed ones
                    continue
                if pd.notna(row[col]):
                    print(f"   {col}: {row[col]}")
            print("-" * 60)
        
        # Summary statistics for poorest performers
        print(f"\nSUMMARY OF POOREST {n_poorest} PERFORMERS:")
        print(f"  Mean Test Pearson: {poorest_performers['test_pearson'].mean():.4f}")
        print(f"  Mean Total Counts: {poorest_performers['total_counts'].mean():,.0f}")
        print(f"  Median Total Counts: {poorest_performers['total_counts'].median():,.0f}")
        
        # Check if there are common characteristics
        if color_by and color_by in df.columns:
            poorest_categories = poorest_performers[color_by].value_counts()
            print(f"\n  Distribution by {color_by}:")
            for category, count in poorest_categories.items():
                percentage = (count / len(poorest_performers)) * 100
                print(f"    {category}: {count} ({percentage:.1f}%)")
        
        print("="*80)
    
    return fig, df, poorest_performers if show_poorest_performers else None

# Additional function to analyze patterns in poor performers
def analyze_poor_performers(ad, n_poorest=10, compare_to_best=True):
    """
    Detailed analysis of poorest performing pseudobulks
    
    Parameters:
    -----------
    ad : AnnData object
        The pseudobulk data
    n_poorest : int
        Number of poorest performers to analyze
    compare_to_best : bool
        Whether to compare with best performers
    """
    import pandas as pd
    import numpy as np
    
    df = ad.obs.copy()
    df = df.dropna(subset=['test_pearson'])
    
    # Get poorest and best performers
    poorest = df.nsmallest(n_poorest, 'test_pearson')
    
    print("="*80)
    print("DETAILED ANALYSIS OF POOR PERFORMERS")
    print("="*80)
    
    if compare_to_best:
        best = df.nlargest(n_poorest, 'test_pearson')
        print(f"\nComparison between {n_poorest} POOREST vs {n_poorest} BEST performers:")
        print("-" * 60)
        
        # Compare key metrics
        metrics_to_compare = ['total_counts', 'n_genes', 'n_cells', 'size_factor']
        
        for metric in metrics_to_compare:
            if metric in df.columns:
                poor_mean = poorest[metric].mean()
                best_mean = best[metric].mean()
                poor_median = poorest[metric].median()
                best_median = best[metric].median()
                
                print(f"\n{metric.upper()}:")
                print(f"  Poorest - Mean: {poor_mean:,.2f}, Median: {poor_median:,.2f}")
                print(f"  Best    - Mean: {best_mean:,.2f}, Median: {best_median:,.2f}")
                print(f"  Ratio (Poor/Best): {poor_mean/best_mean:.2f}")
        
        # Compare categorical variables
        categorical_vars = ['cell_type', 'tissue', 'organ', 'timepoint', 'disease']
        
        for var in categorical_vars:
            if var in df.columns:
                print(f"\n{var.upper()} DISTRIBUTION:")
                poor_dist = poorest[var].value_counts(normalize=True) * 100
                best_dist = best[var].value_counts(normalize=True) * 100
                
                # Combine and compare
                comparison_df = pd.DataFrame({
                    'Poorest (%)': poor_dist,
                    'Best (%)': best_dist
                }).fillna(0)
                
                print(comparison_df.round(1))
    
    # Look for patterns in poor performers
    print(f"\n\nPATTERN ANALYSIS FOR POOREST {n_poorest} PERFORMERS:")
    print("-" * 60)
    
    # Check if certain categories are overrepresented
    categorical_vars = ['cell_type', 'tissue', 'organ', 'timepoint', 'disease']
    
    for var in categorical_vars:
        if var in df.columns:
            # Calculate representation in poor performers vs overall
            overall_dist = df[var].value_counts(normalize=True) * 100
            poor_dist = poorest[var].value_counts(normalize=True) * 100
            
            # Find categories that are overrepresented in poor performers
            overrepresented = []
            for category in poor_dist.index:
                if category in overall_dist.index:
                    ratio = poor_dist[category] / overall_dist[category]
                    if ratio > 1.5:  # More than 50% overrepresented
                        overrepresented.append((category, ratio, poor_dist[category], overall_dist[category]))
            
            if overrepresented:
                print(f"\n{var.upper()} categories overrepresented in poor performers:")
                for cat, ratio, poor_pct, overall_pct in overrepresented:
                    print(f"  {cat}: {ratio:.1f}x overrepresented ({poor_pct:.1f}% vs {overall_pct:.1f}% overall)")
    
    return poorest, best if compare_to_best else None

# Example usage:
# result = plot_total_counts_vs_test_pearson(ad, model_name="Your Model", show_poorest_performers=True, n_poorest=5)
# fig, df, poorest = result

# For detailed analysis:
# poorest_detailed, best_detailed = analyze_poor_performers(ad, n_poorest=10, compare_to_best=True)

In [ ]:
def add_count_corrected_metrics(ad):
    """
    Add count-corrected performance metrics to remove total count bias
    
    Parameters:
    -----------
    ad : AnnData object
        The pseudobulk data
    
    Returns:
    --------
    ad : AnnData object with new columns added
    """
    import numpy as np
    from sklearn.linear_model import LinearRegression
    import pandas as pd
    
    df = ad.obs.copy()
    df = df.dropna(subset=['total_counts', 'test_pearson'])
    
    # Method 1: Linear regression residuals
    log_counts = np.log10(df['total_counts']).values.reshape(-1, 1)
    
    # Fit regression: pearson ~ log10(total_counts)
    reg = LinearRegression()
    reg.fit(log_counts, df['test_pearson'])
    
    # Calculate residuals (count-corrected performance)
    predicted_pearson = reg.predict(log_counts)
    residuals = df['test_pearson'] - predicted_pearson
    
    # Add to dataframe
    df['predicted_pearson_from_counts'] = predicted_pearson
    df['count_corrected_pearson'] = residuals
    
    # Method 2: Percentile-based correction within count bins
    # Bin pseudobulks by total counts and calculate percentiles within bins
    df['count_bin'] = pd.qcut(df['total_counts'], q=10, labels=False, duplicates='drop')
    df['pearson_percentile_in_count_bin'] = df.groupby('count_bin')['test_pearson'].rank(pct=True)
    
    # Method 3: Z-score within count bins
    df['pearson_zscore_in_count_bin'] = df.groupby('count_bin')['test_pearson'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    
    # Add back to AnnData
    for col in ['predicted_pearson_from_counts', 'count_corrected_pearson', 
                'count_bin', 'pearson_percentile_in_count_bin', 'pearson_zscore_in_count_bin']:
        ad.obs[col] = np.nan
        ad.obs.loc[df.index, col] = df[col]
    
    # Print summary
    print("Added count-corrected metrics:")
    print(f"  count_corrected_pearson: Residuals from log(counts) ~ pearson regression")
    print(f"  pearson_percentile_in_count_bin: Percentile rank within count deciles")
    print(f"  pearson_zscore_in_count_bin: Z-score within count deciles")
    
    # Show correlation reduction
    original_corr = np.corrcoef(df['total_counts'], df['test_pearson'])[0,1]
    corrected_corr = np.corrcoef(df['total_counts'], df['count_corrected_pearson'])[0,1]
    
    print(f"\nCorrelation with total_counts:")
    print(f"  Original test_pearson: {original_corr:.3f}")
    print(f"  Count-corrected pearson: {corrected_corr:.3f}")
    
    return ad

def compare_models_count_corrected(model_ads, model_names, correction_method='count_corrected_pearson'):
    """
    Compare models using count-corrected metrics
    
    Parameters:
    -----------
    model_ads : list
        List of AnnData objects for different models
    model_names : list
        Names of the models
    correction_method : str
        Which correction method to use
    """
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Combine data from all models
    combined_data = []
    
    for ad, name in zip(model_ads, model_names):
        df = ad.obs.copy()
        df['model'] = name
        combined_data.append(df)
    
    all_data = pd.concat(combined_data, ignore_index=True)
    all_data = all_data.dropna(subset=[correction_method, 'test_pearson'])
    
    # Analysis by cell type/tissue
    categorical_vars = ['cell_type', 'tissue', 'timepoint']
    
    for var in categorical_vars:
        if var in all_data.columns:
            print(f"\n{'='*60}")
            print(f"COUNT-CORRECTED ANALYSIS BY {var.upper()}")
            print(f"{'='*60}")
            
            # Calculate mean performance for each category and model
            summary = all_data.groupby([var, 'model']).agg({
                'test_pearson': ['mean', 'std', 'count'],
                correction_method: ['mean', 'std']
            }).round(4)
            
            # Flatten column names
            summary.columns = ['_'.join(col).strip() for col in summary.columns]
            summary = summary.reset_index()
            
            # Find categories with biggest differences between models
            pivot_original = all_data.pivot_table(
                values='test_pearson', 
                index=var, 
                columns='model', 
                aggfunc='mean'
            )
            
            pivot_corrected = all_data.pivot_table(
                values=correction_method, 
                index=var, 
                columns='model', 
                aggfunc='mean'
            )
            
            print(f"\nOriginal test_pearson by {var}:")
            print(pivot_original.round(4))
            
            print(f"\nCount-corrected pearson by {var}:")
            print(pivot_corrected.round(4))
            
            # Calculate differences between models (assuming first model is baseline)
            if len(model_names) >= 2:
                baseline_model = model_names[0]
                for comparison_model in model_names[1:]:
                    if baseline_model in pivot_corrected.columns and comparison_model in pivot_corrected.columns:
                        diff_original = pivot_original[comparison_model] - pivot_original[baseline_model]
                        diff_corrected = pivot_corrected[comparison_model] - pivot_corrected[baseline_model]
                        
                        print(f"\nDifference ({comparison_model} - {baseline_model}):")
                        comparison_df = pd.DataFrame({
                            f'{var}': diff_original.index,
                            'Original_Diff': diff_original.values,
                            'Corrected_Diff': diff_corrected.values,
                            'Change_in_Benefit': diff_corrected.values - diff_original.values
                        })
                        
                        # Sort by biggest change in benefit
                        comparison_df = comparison_df.sort_values('Change_in_Benefit', ascending=False)
                        print(comparison_df.round(4))
                        
                        # Highlight categories that benefit more/less after correction
                        print(f"\nCategories that benefit MORE after count correction:")
                        improved = comparison_df[comparison_df['Change_in_Benefit'] > 0.01]
                        for _, row in improved.iterrows():
                            print(f"  {row[var]}: {row['Change_in_Benefit']:.4f} improvement in relative benefit")
                        
                        print(f"\nCategories that benefit LESS after count correction:")
                        diminished = comparison_df[comparison_df['Change_in_Benefit'] < -0.01]
                        for _, row in diminished.iterrows():
                            print(f"  {row[var]}: {row['Change_in_Benefit']:.4f} reduction in relative benefit")

def plot_count_correction_comparison(ad, model_name="Model"):
    """
    Visualize the effect of count correction
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    df = ad.obs.dropna(subset=['total_counts', 'test_pearson', 'count_corrected_pearson'])
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{model_name}: Count Correction Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Original correlation
    ax1 = axes[0, 0]
    ax1.scatter(df['total_counts'], df['test_pearson'], alpha=0.6, s=30)
    ax1.set_xscale('log')
    ax1.set_xlabel('Total Counts (log)')
    ax1.set_ylabel('Test Pearson (Original)')
    ax1.set_title('Original: Strong Correlation')
    corr_orig = np.corrcoef(df['total_counts'], df['test_pearson'])[0,1]
    ax1.text(0.05, 0.95, f'r = {corr_orig:.3f}', transform=ax1.transAxes, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 2: Corrected correlation
    ax2 = axes[0, 1]
    ax2.scatter(df['total_counts'], df['count_corrected_pearson'], alpha=0.6, s=30, color='red')
    ax2.set_xscale('log')
    ax2.set_xlabel('Total Counts (log)')
    ax2.set_ylabel('Count-Corrected Pearson')
    ax2.set_title('Corrected: Reduced Correlation')
    corr_corr = np.corrcoef(df['total_counts'], df['count_corrected_pearson'])[0,1]
    ax2.text(0.05, 0.95, f'r = {corr_corr:.3f}', transform=ax2.transAxes,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 3: Original vs Corrected
    ax3 = axes[0, 2]
    ax3.scatter(df['test_pearson'], df['count_corrected_pearson'], alpha=0.6, s=30, color='green')
    ax3.set_xlabel('Original Test Pearson')
    ax3.set_ylabel('Count-Corrected Pearson')
    ax3.set_title('Original vs Corrected')
    ax3.plot([df['test_pearson'].min(), df['test_pearson'].max()], [0, 0], 'r--', alpha=0.5)
    
    # Plot 4: Distribution comparison
    ax4 = axes[1, 0]
    ax4.hist(df['test_pearson'], bins=30, alpha=0.7, label='Original', density=True)
    ax4.hist(df['count_corrected_pearson'], bins=30, alpha=0.7, label='Corrected', density=True)
    ax4.set_xlabel('Pearson Correlation')
    ax4.set_ylabel('Density')
    ax4.set_title('Distribution Comparison')
    ax4.legend()
    
    # Plot 5: Percentile method
    ax5 = axes[1, 1]
    ax5.scatter(df['total_counts'], df['pearson_percentile_in_count_bin'], alpha=0.6, s=30, color='purple')
    ax5.set_xscale('log')
    ax5.set_xlabel('Total Counts (log)')
    ax5.set_ylabel('Percentile within Count Bin')
    ax5.set_title('Percentile-Based Correction')
    
    # Plot 6: Z-score method
    ax6 = axes[1, 2]
    ax6.scatter(df['total_counts'], df['pearson_zscore_in_count_bin'], alpha=0.6, s=30, color='orange')
    ax6.set_xscale('log')
    ax6.set_xlabel('Total Counts (log)')
    ax6.set_ylabel('Z-score within Count Bin')
    ax6.set_title('Z-score-Based Correction')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Usage example:
def run_count_corrected_analysis(model_ads, model_names):
    """
    Complete workflow for count-corrected analysis
    """
    print("Adding count-corrected metrics to all models...")
    
    # Add corrected metrics to each model
    corrected_ads = []
    for ad, name in zip(model_ads, model_names):
        print(f"\nProcessing {name}...")
        corrected_ad = add_count_corrected_metrics(ad)
        corrected_ads.append(corrected_ad)
        
        # Show visualization for first model
        if name == model_names[0]:
            plot_count_correction_comparison(corrected_ad, name)
    
    # Compare models using corrected metrics
    print(f"\n{'='*80}")
    print("COMPARING MODELS WITH COUNT CORRECTION")
    print(f"{'='*80}")
    
    compare_models_count_corrected(corrected_ads, model_names, 'count_corrected_pearson')
    
    return corrected_ads

In [ ]:
# ad = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20473753/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20473753-0_20250616_Human_Decima_0.h5ad")
# ad = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-0_20250529_Human_Borzoi_0.h5ad")
ad_random = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed42/version_0/data_out_decima_random_lr3e-06_seed42_20250619_Random_0.h5ad")
ad_decima = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep0_lr3e-05_seed42_20250619_Human_Decima_0.h5ad")


# model_paths = [
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/0/task_0/lr_1.154578199918017e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-0_20250529_Random_0.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/5/task_5/lr_7.116198906552475e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-5_20250529_Random_1.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/1/task_1/lr_1.534060288605152e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-1_20250529_Random_2.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/2/task_2/lr_2.9680669393835383e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-2_20250529_Random_3.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/3/task_3/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-3_20250529_Human_Borzoi_3.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/2/task_2/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-2_20250529_Human_Borzoi_2.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/1/task_1/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-1_20250529_Human_Borzoi_1.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-0_20250529_Human_Borzoi_0.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-0_20250616_Mouse_Borzoi_0.h5ad",
#     # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/1/task_1/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-1_20250616_Mouse_Borzoi_1.h5ad",
#     # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/2/task_2/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-2_20250616_Mouse_Borzoi_2.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20490451/3/task_3/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20490451-3_20250616_Mouse_Borzoi_3.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20473753/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20473753-0_20250616_Human_Decima_0.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491529/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491529-0_20250616_Human_Decima_1.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491962/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491962-0_20250616_Human_Decima_2.h5ad",
#     "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491359/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_20491359-0_20250616_Human_Decima_3.h5ad"

In [ ]:
# Assuming you have multiple model AnnData objects
model_ads = [ad_random, ad_decima]  # Your models
model_names = ['Random', 'Human_Decima']

# Run the analysis
corrected_ads = run_count_corrected_analysis(model_ads, model_names)

# You can also run individual functions:
# Add correction to single model
#ad_corrected = add_count_corrected_metrics(your_ad)

# Visualize the correction
#plot_count_correction_comparison(ad_corrected, "Your Model")

In [ ]:
corrected_ads[0].obs

In [ ]:
# Basic usage with poorest performers highlighted
fig, df, poorest = plot_total_counts_vs_test_pearson(
    ad_decima, 
    model_name="Human Decima Rep 0",
    color_by="cell_type",  # This will help identify patterns
    show_poorest_performers=False,
    n_poorest=5
)

# # For more detailed analysis
# poorest_detailed, best_detailed = analyze_poor_performers(
#     ad, 
#     n_poorest=25, 
#     compare_to_best=True
#)

In [ ]:
df.sort_values(by='test_pearson', ascending=False)

In [ ]:
def plot_focused_poor_vs_best_analysis(ad, n_performers=25, figsize=(18, 12)):
    """
    Focused visual comparison of poorest vs best performing pseudobulks
    Now includes both poor and best performing cell type analysis
    
    Parameters:
    -----------
    ad : AnnData object
        The pseudobulk data
    n_performers : int
        Number of poorest and best performers to compare
    figsize : tuple
        Figure size (increased to accommodate new layout)
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    
    # Prepare data
    df = ad.obs.copy().dropna(subset=['test_pearson'])
    poorest = df.nsmallest(n_performers, 'test_pearson').copy()
    best = df.nlargest(n_performers, 'test_pearson').copy()
    
    # DIAGNOSTIC: Show what cell types are in the poorest and best performers
    print(f"\n📊 BREAKDOWN OF {n_performers} POOREST PERFORMERS BY CELL TYPE:")
    print("="*60)
    poor_celltype_counts = poorest['zebrafish_anatomy_ontology_class_fine'].value_counts()
    for celltype, count in poor_celltype_counts.items():
        print(f"  • {celltype}: {count} poor performers")
    
    print(f"\n📊 BREAKDOWN OF {n_performers} BEST PERFORMERS BY CELL TYPE:")
    print("="*60)
    best_celltype_counts = best['zebrafish_anatomy_ontology_class_fine'].value_counts()
    for celltype, count in best_celltype_counts.items():
        print(f"  • {celltype}: {count} best performers")
    
    # Create the subplot layout (3x2 grid for better cell type analysis)
    fig, axes = plt.subplots(3, 2, figsize=figsize)
    fig.suptitle(f'Poor vs Best Performers Analysis (n={n_performers} each)', 
                 fontsize=16, fontweight='bold')
    
    # Add performance category for violin plot
    poorest['performance'] = 'Poorest'
    best['performance'] = 'Best'
    comparison_df = pd.concat([poorest, best])
    
    # 1. TOTAL COUNTS COMPARISON (Top Left)
    ax1 = axes[0, 0]
    
    # Create violin plot for total counts
    violin_parts = ax1.violinplot([poorest['total_counts'], best['total_counts']], 
                                  positions=[0, 1], showmeans=False, showmedians=False)
    
    # Color the violins
    violin_parts['bodies'][0].set_color('red')
    violin_parts['bodies'][0].set_alpha(0.7)
    violin_parts['bodies'][1].set_color('green') 
    violin_parts['bodies'][1].set_alpha(0.7)
    
    # Add mean diamonds
    poor_mean = poorest['total_counts'].mean()
    best_mean = best['total_counts'].mean()
    
    ax1.scatter(0, poor_mean, color='white', s=150, zorder=10, marker='D', 
               edgecolor='black', linewidth=2, label='Mean')
    ax1.scatter(1, best_mean, color='white', s=150, zorder=10, marker='D', 
               edgecolor='black', linewidth=2)
    
    # Add mean labels
    ax1.text(0, poor_mean, f'{poor_mean:,.0f}', ha='center', va='center', 
           fontweight='bold', fontsize=10)
    ax1.text(1, best_mean, f'{best_mean:,.0f}', ha='center', va='center', 
           fontweight='bold', fontsize=10)
    
    ax1.set_yscale('log')
    ax1.set_title('Total Counts Distribution')
    ax1.set_xticks([0, 1])
    ax1.set_xticklabels(['Poorest', 'Best'])
    ax1.set_ylabel('Total Counts (log scale)')
    ax1.grid(True, alpha=0.3)
    
    # Calculate and display ratio
    ratio = poor_mean / best_mean
    ax1.text(0.5, 0.95, f'Poor/Best Ratio: {ratio:.2f}\n({1/ratio:.1f}x lower)', 
           transform=ax1.transAxes, ha='center', va='top', 
           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8), fontweight='bold')
    
    # 2. TIMEPOINT DISTRIBUTION (Top Right)
    ax2 = axes[0, 1]
    
    # Define correct temporal order
    timepoint_order = ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf', '24hpf', '2dpf', '3dpf', '5dpf', '10dpf']
    existing_timepoints = [tp for tp in timepoint_order if tp in df['timepoint'].unique()]
    
    # Calculate percentages for each timepoint in correct order
    timepoint_comparison = []
    for timepoint in existing_timepoints:
        poor_count = (poorest['timepoint'] == timepoint).sum()
        best_count = (best['timepoint'] == timepoint).sum()
        overall_count = (df['timepoint'] == timepoint).sum()
        
        poor_pct = (poor_count / len(poorest)) * 100
        best_pct = (best_count / len(best)) * 100
        overall_pct = (overall_count / len(df)) * 100
        
        timepoint_comparison.append({
            'timepoint': timepoint,
            'Poorest': poor_pct,
            'Best': best_pct,
            'Overall': overall_pct
        })
    
    timepoint_df = pd.DataFrame(timepoint_comparison)
    
    # Create grouped bar plot
    x = np.arange(len(timepoint_df))
    width = 0.25
    
    bars1 = ax2.bar(x - width, timepoint_df['Poorest'], width, 
                   label='Poorest', color='red', alpha=0.7)
    bars2 = ax2.bar(x, timepoint_df['Best'], width, 
                   label='Best', color='green', alpha=0.7)
    bars3 = ax2.bar(x + width, timepoint_df['Overall'], width, 
                   label='Overall', color='gray', alpha=0.5)
    
    ax2.set_xlabel('Timepoint (Temporal Order)')
    ax2.set_ylabel('Percentage (%)')
    ax2.set_title('Timepoint Distribution')
    ax2.set_xticks(x)
    ax2.set_xticklabels(timepoint_df['timepoint'], rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add percentage labels on bars (only for non-zero values)
    for bars, values in [(bars1, timepoint_df['Poorest']), (bars2, timepoint_df['Best'])]:
        for bar, val in zip(bars, values):
            if val > 2:  # Only label if > 2%
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                        f'{val:.0f}%', ha='center', va='bottom', fontsize=8)
    
    # 3. POOR PERFORMING CELL TYPES (Middle Left)
    ax3 = axes[1, 0]
    
    # Only analyze cell types that have poor performers
    poor_celltypes = poorest['zebrafish_anatomy_ontology_class_fine'].unique()
    poor_celltype_analysis = []
    
    for celltype in poor_celltypes:
        total_count = (df['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
        poor_count = (poorest['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
        
        if total_count > 0:
            failure_rate = (poor_count / total_count) * 100
            poor_celltype_analysis.append({
                'celltype': celltype,
                'total_count': total_count,
                'poor_count': poor_count,
                'failure_rate': failure_rate
            })
    
    poor_celltype_df = pd.DataFrame(poor_celltype_analysis).sort_values('failure_rate', ascending=True)
    
    # Horizontal bar plot for poor performers
    y_pos = np.arange(len(poor_celltype_df))
    colors = ['darkred' if x > 50 else 'red' if x > 30 else 'orange' if x > 20 else 'coral' 
              for x in poor_celltype_df['failure_rate']]
    
    bars = ax3.barh(y_pos, poor_celltype_df['failure_rate'], color=colors, alpha=0.7)
    
    ax3.set_yticks(y_pos)
    truncated_names = [ct[:20] + '...' if len(ct) > 20 else ct for ct in poor_celltype_df['celltype']]
    ax3.set_yticklabels(truncated_names, fontsize=9)
    ax3.set_xlabel('Failure Rate (%)')
    ax3.set_title('Cell Types with Poor Performers')
    ax3.grid(True, alpha=0.3)
    
    # Add failure rate values and counts
    for i, (bar, row) in enumerate(zip(bars, poor_celltype_df.itertuples())):
        ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
               f'{row.failure_rate:.1f}%\n({row.poor_count}/{row.total_count})', 
               ha='left', va='center', fontsize=8, fontweight='bold')
    
    # 4. BEST PERFORMING CELL TYPES (Middle Right)
    ax4 = axes[1, 1]
    
    # Only analyze cell types that have best performers
    best_celltypes = best['zebrafish_anatomy_ontology_class_fine'].unique()
    best_celltype_analysis = []
    
    for celltype in best_celltypes:
        total_count = (df['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
        best_count = (best['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
        
        if total_count > 0:
            success_rate = (best_count / total_count) * 100
            best_celltype_analysis.append({
                'celltype': celltype,
                'total_count': total_count,
                'best_count': best_count,
                'success_rate': success_rate
            })
    
    best_celltype_df = pd.DataFrame(best_celltype_analysis).sort_values('success_rate', ascending=True)
    
    # Horizontal bar plot for best performers
    y_pos = np.arange(len(best_celltype_df))
    colors = ['darkgreen' if x > 50 else 'green' if x > 30 else 'lightgreen' if x > 20 else 'palegreen' 
              for x in best_celltype_df['success_rate']]
    
    bars = ax4.barh(y_pos, best_celltype_df['success_rate'], color=colors, alpha=0.7)
    
    ax4.set_yticks(y_pos)
    truncated_names = [ct[:20] + '...' if len(ct) > 20 else ct for ct in best_celltype_df['celltype']]
    ax4.set_yticklabels(truncated_names, fontsize=9)
    ax4.set_xlabel('Success Rate (%)')
    ax4.set_title('Cell Types with Best Performers')
    ax4.grid(True, alpha=0.3)
    
    # Add success rate values and counts
    for i, (bar, row) in enumerate(zip(bars, best_celltype_df.itertuples())):
        ax4.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
               f'{row.success_rate:.1f}%\n({row.best_count}/{row.total_count})', 
               ha='left', va='center', fontsize=8, fontweight='bold')
    
    # 5. PERFORMANCE DISTRIBUTION (Bottom Left)
    ax5 = axes[2, 0]
    
    # Plot distribution of test_pearson
    ax5.hist(df['test_pearson'], bins=40, alpha=0.6, color='lightblue', 
            edgecolor='black', label=f'All pseudobulks (n={len(df)})')
    
    # Highlight poor and best regions
    poor_threshold = poorest['test_pearson'].max()
    best_threshold = best['test_pearson'].min()
    
    ax5.axvline(poor_threshold, color='red', linestyle='--', linewidth=3, 
               label=f'Poor threshold ({poor_threshold:.3f})')
    ax5.axvline(best_threshold, color='green', linestyle='--', linewidth=3, 
               label=f'Best threshold ({best_threshold:.3f})')
    
    # Fill regions
    ax5.axvspan(df['test_pearson'].min(), poor_threshold, color='red', alpha=0.3, 
               label=f'Poorest {n_performers}')
    ax5.axvspan(best_threshold, df['test_pearson'].max(), color='green', alpha=0.3, 
               label=f'Best {n_performers}')
    
    ax5.set_xlabel('Test Pearson Correlation')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Performance Distribution')
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)
    
    # Add summary statistics
    ax5.text(0.02, 0.95, f'Mean: {df["test_pearson"].mean():.3f}\nStd: {df["test_pearson"].std():.3f}', 
            transform=ax5.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 6. CELL TYPE COMPARISON SUMMARY (Bottom Right)
    ax6 = axes[2, 1]
    ax6.axis('off')
    
    # Create summary text
    summary_text = f"""
CELL TYPE PERFORMANCE SUMMARY

🔴 POOR PERFORMERS:
• {len(poor_celltype_df)} cell types have poor performers
• Worst: {poor_celltype_df.iloc[-1]['celltype']} ({poor_celltype_df.iloc[-1]['failure_rate']:.1f}% failure)
• Average failure rate: {poor_celltype_df['failure_rate'].mean():.1f}%

🟢 BEST PERFORMERS:  
• {len(best_celltype_df)} cell types have best performers
• Best: {best_celltype_df.iloc[-1]['celltype']} ({best_celltype_df.iloc[-1]['success_rate']:.1f}% success)
• Average success rate: {best_celltype_df['success_rate'].mean():.1f}%

📊 OVERLAP:
• Cell types in both groups: {len(set(poor_celltypes) & set(best_celltypes))}
• Poor-only cell types: {len(set(poor_celltypes) - set(best_celltypes))}
• Best-only cell types: {len(set(best_celltypes) - set(poor_celltypes))}
    """
    
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print key findings
    print("\n🔍 KEY FINDINGS:")
    print("-" * 40)
    
    # Total counts finding
    print(f"💰 SEQUENCING DEPTH: Poor performers have {1/ratio:.1f}x lower total counts")
    print(f"   (Poor: {poor_mean:,.0f} vs Best: {best_mean:,.0f})")
    
    # Timepoint findings
    print(f"\n⏰ TIMEPOINT PATTERNS:")
    for _, row in timepoint_df.iterrows():
        if row['Poorest'] > row['Overall'] * 1.5:
            overrep = row['Poorest'] / row['Overall']
            stage_context = "late development" if "dpf" in row['timepoint'] else "early development"
            print(f"   • {row['timepoint']} ({stage_context}): {overrep:.1f}x overrepresented in poor performers")
    
    # Cell type findings
    print(f"\n🧬 PROBLEMATIC CELL TYPES (>50% failure rate):")
    high_failure = poor_celltype_df[poor_celltype_df['failure_rate'] > 50].sort_values('failure_rate', ascending=False)
    for _, row in high_failure.iterrows():
        print(f"   • {row['celltype']}: {row['failure_rate']:.1f}% failure rate ({row['poor_count']}/{row['total_count']} examples)")
    
    print(f"\n🌟 EXCELLENT CELL TYPES (>50% success rate):")
    high_success = best_celltype_df[best_celltype_df['success_rate'] > 50].sort_values('success_rate', ascending=False)
    for _, row in high_success.iterrows():
        print(f"   • {row['celltype']}: {row['success_rate']:.1f}% success rate ({row['best_count']}/{row['total_count']} examples)")
    
    return fig, poorest, best, poor_celltype_df, best_celltype_df

# Example usage:
# fig, poorest, best, poor_celltypes, best_celltypes = plot_focused_poor_vs_best_analysis(ad, n_performers=25)

In [ ]:

# Example usage:
fig, poorest, best, poor_celltypes, best_celltypes = plot_focused_poor_vs_best_analysis(ad_random, n_performers=15)

In [ ]:
def load_and_analyze_model_replicates(model_paths, model_names, n_performers=25, figsize=(20, 14)):
    """
    Load and analyze model replicates from paths and names
    Focused on poor vs best performer analysis with timepoint and cell type breakdowns
    
    Parameters:
    -----------
    model_paths : list
        List of file paths to model results
    model_names : list  
        List of corresponding model names
    n_performers : int
        Number of poorest and best performers to analyze per model
    figsize : tuple
        Figure size
    """
    import scanpy as sc
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("="*80)
    print("LOADING AND ANALYZING MODEL REPLICATES")
    print("="*80)
    
    # Group models by type (Random vs Human_Borzoi)
    model_groups = {}
    
    for path, name in zip(model_paths, model_names):
        # Extract model type from name
        if 'Random' in name:
            model_type = 'Random Init'
        elif 'Human_Borzoi' in name:
            model_type = 'Human Borzoi'
        else:
            model_type = 'Unknown'
        
        if model_type not in model_groups:
            model_groups[model_type] = []
        
        print(f"Loading {name}...")
        try:
            ad = sc.read_h5ad(path)
            model_groups[model_type].append((ad, name))
            print(f"  ✓ Loaded {name}: {ad.n_obs} pseudobulks")
        except Exception as e:
            print(f"  ✗ Failed to load {name}: {e}")
    
    print(f"\nFound model groups: {list(model_groups.keys())}")
    for group, models in model_groups.items():
        print(f"  • {group}: {len(models)} replicates")
    
    # Aggregate data for each model type
    aggregated_data = {}
    
    for model_type, replicate_list in model_groups.items():
        print(f"\n📊 Processing {model_type}: {len(replicate_list)} replicates")
        
        # Combine all replicates for this model type
        combined_obs_list = []
        for ad, name in replicate_list:
            df_rep = ad.obs.copy().dropna(subset=['test_pearson'])
            df_rep['replicate'] = name
            df_rep['model_type'] = model_type
            combined_obs_list.append(df_rep)
        
        if not combined_obs_list:
            print(f"  ⚠️  No valid data for {model_type}")
            continue
            
        combined_df = pd.concat(combined_obs_list, ignore_index=True)
        
        # Get poor and best performers for this model (across all replicates)
        poorest_model = combined_df.nsmallest(n_performers, 'test_pearson').copy()
        best_model = combined_df.nlargest(n_performers, 'test_pearson').copy()
        
        poorest_model['performance'] = 'Poorest'
        best_model['performance'] = 'Best'
        
        aggregated_data[model_type] = {
            'combined_df': combined_df,
            'poorest': poorest_model,
            'best': best_model,
            'replicates': replicate_list
        }
        
        print(f"   • Total pseudobulks: {len(combined_df)}")
        print(f"   • Mean performance: {combined_df['test_pearson'].mean():.3f}")
        print(f"   • Std performance: {combined_df['test_pearson'].std():.3f}")
    
    # Create the subplot layout (3x2 grid)
    fig, axes = plt.subplots(3, 2, figsize=figsize)
    fig.suptitle(f'Model Comparison: Random Init vs Human Borzoi\n(n={n_performers} worst/best per model type)', 
                 fontsize=16, fontweight='bold')
    
    model_types = list(aggregated_data.keys())
    
    # 1. TOTAL COUNTS COMPARISON BY MODEL (Top Left)
    ax1 = axes[0, 0]
    
    if len(model_types) >= 2:
        poor_data = [aggregated_data[model]['poorest']['total_counts'] for model in model_types]
        best_data = [aggregated_data[model]['best']['total_counts'] for model in model_types]
        
        positions_poor = [i*2 for i in range(len(model_types))]
        positions_best = [i*2 + 0.8 for i in range(len(model_types))]
        
        vp_poor = ax1.violinplot(poor_data, positions=positions_poor, widths=0.6)
        vp_best = ax1.violinplot(best_data, positions=positions_best, widths=0.6)
        
        # Color the violins
        for pc in vp_poor['bodies']:
            pc.set_facecolor('red')
            pc.set_alpha(0.7)
        for pc in vp_best['bodies']:
            pc.set_facecolor('green')
            pc.set_alpha(0.7)
        
        # Add means
        for i, model in enumerate(model_types):
            poor_mean = aggregated_data[model]['poorest']['total_counts'].mean()
            best_mean = aggregated_data[model]['best']['total_counts'].mean()
            
            ax1.scatter(i*2, poor_mean, color='white', s=100, marker='D', 
                       edgecolor='black', linewidth=2, zorder=10)
            ax1.scatter(i*2 + 0.8, best_mean, color='white', s=100, marker='D', 
                       edgecolor='black', linewidth=2, zorder=10)
            
            ax1.text(i*2, poor_mean, f'{poor_mean:,.0f}', ha='center', va='center', 
                   fontweight='bold', fontsize=8)
            ax1.text(i*2 + 0.8, best_mean, f'{best_mean:,.0f}', ha='center', va='center', 
                   fontweight='bold', fontsize=8)
        
        ax1.set_yscale('log')
        ax1.set_title('Total Counts by Model Type')
        ax1.set_xticks([i*2 + 0.4 for i in range(len(model_types))])
        ax1.set_xticklabels([mt.replace(' ', '\n') for mt in model_types], fontsize=10)
        ax1.set_ylabel('Total Counts (log scale)')
        ax1.grid(True, alpha=0.3)
        
        # Create custom legend
        ax1.scatter([], [], color='red', alpha=0.7, s=100, label='Poorest')
        ax1.scatter([], [], color='green', alpha=0.7, s=100, label='Best')
        ax1.legend()
    
    # 2. PERFORMANCE COMPARISON BY MODEL (Top Right)
    ax2 = axes[0, 1]
    
    performance_data = []
    for model_type, data in aggregated_data.items():
        for _, row in data['poorest'].iterrows():
            performance_data.append({'Model': model_type, 'Group': 'Poorest', 'Performance': row['test_pearson']})
        for _, row in data['best'].iterrows():
            performance_data.append({'Model': model_type, 'Group': 'Best', 'Performance': row['test_pearson']})
    
    if performance_data:
        perf_df = pd.DataFrame(performance_data)
        sns.boxplot(data=perf_df, x='Model', y='Performance', hue='Group', ax=ax2, palette=['red', 'green'])
        ax2.set_title('Performance Distribution by Model')
        ax2.set_ylabel('Test Pearson Correlation')
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)
    
    # 3. TIMEPOINT ENRICHMENT FOR POOR PERFORMERS (Middle Left)
    ax3 = axes[1, 0]
    
    timepoint_order = ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf', '24hpf', '2dpf', '3dpf', '5dpf', '10dpf']
    
    poor_timepoint_data = []
    for model_type, data in aggregated_data.items():
        combined_df = data['combined_df']
        poorest = data['poorest']
        
        existing_timepoints = [tp for tp in timepoint_order if tp in combined_df['timepoint'].unique()]
        
        for timepoint in existing_timepoints:
            poor_count = (poorest['timepoint'] == timepoint).sum()
            total_poor = len(poorest)
            overall_count = (combined_df['timepoint'] == timepoint).sum()
            total_overall = len(combined_df)
            
            poor_pct = (poor_count / total_poor) * 100 if total_poor > 0 else 0
            overall_pct = (overall_count / total_overall) * 100 if total_overall > 0 else 0
            
            poor_timepoint_data.append({
                'model': model_type,
                'timepoint': timepoint,
                'poor_count': poor_count,
                'poor_pct': poor_pct,
                'overall_pct': overall_pct,
                'enrichment': poor_pct / overall_pct if overall_pct > 0 else 0
            })
    
    if poor_timepoint_data:
        poor_timepoint_df = pd.DataFrame(poor_timepoint_data)
        timepoints_to_show = poor_timepoint_df['timepoint'].unique()
        x = np.arange(len(timepoints_to_show))
        width = 0.35
        
        for i, model_type in enumerate(model_types):
            model_data = poor_timepoint_df[poor_timepoint_df['model'] == model_type]
            enrichments = [model_data[model_data['timepoint'] == tp]['enrichment'].iloc[0] 
                          if len(model_data[model_data['timepoint'] == tp]) > 0 else 0 
                          for tp in timepoints_to_show]
            
            color = 'red' if 'Random' in model_type else 'blue'
            bars = ax3.bar(x + i*width, enrichments, width, label=model_type, alpha=0.7, color=color)
            
            # Add count labels on bars
            for j, (bar, tp) in enumerate(zip(bars, timepoints_to_show)):
                model_tp_data = model_data[model_data['timepoint'] == tp]
                if len(model_tp_data) > 0 and bar.get_height() > 0.1:
                    count = model_tp_data['poor_count'].iloc[0]
                    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                           f'{count}', ha='center', va='bottom', fontsize=7)
        
        ax3.set_xlabel('Timepoint')
        ax3.set_ylabel('Enrichment in Poor Performers')
        ax3.set_title('Timepoint Enrichment: Poor Performers')
        ax3.set_xticks(x + width/2)
        ax3.set_xticklabels(timepoints_to_show, rotation=45)
        ax3.axhline(y=1, color='black', linestyle='--', alpha=0.5)
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4. TIMEPOINT ENRICHMENT FOR BEST PERFORMERS (Middle Right)
    ax4 = axes[1, 1]
    
    best_timepoint_data = []
    for model_type, data in aggregated_data.items():
        combined_df = data['combined_df']
        best = data['best']
        
        existing_timepoints = [tp for tp in timepoint_order if tp in combined_df['timepoint'].unique()]
        
        for timepoint in existing_timepoints:
            best_count = (best['timepoint'] == timepoint).sum()
            total_best = len(best)
            overall_count = (combined_df['timepoint'] == timepoint).sum()
            total_overall = len(combined_df)
            
            best_pct = (best_count / total_best) * 100 if total_best > 0 else 0
            overall_pct = (overall_count / total_overall) * 100 if total_overall > 0 else 0
            
            best_timepoint_data.append({
                'model': model_type,
                'timepoint': timepoint,
                'best_count': best_count,
                'best_pct': best_pct,
                'overall_pct': overall_pct,
                'enrichment': best_pct / overall_pct if overall_pct > 0 else 0
            })
    
    if best_timepoint_data:
        best_timepoint_df = pd.DataFrame(best_timepoint_data)
        timepoints_to_show = best_timepoint_df['timepoint'].unique()
        x = np.arange(len(timepoints_to_show))
        width = 0.35
        
        for i, model_type in enumerate(model_types):
            model_data = best_timepoint_df[best_timepoint_df['model'] == model_type]
            enrichments = [model_data[model_data['timepoint'] == tp]['enrichment'].iloc[0] 
                          if len(model_data[model_data['timepoint'] == tp]) > 0 else 0 
                          for tp in timepoints_to_show]
            
            color = 'darkgreen' if 'Random' in model_type else 'darkblue'
            bars = ax4.bar(x + i*width, enrichments, width, label=model_type, alpha=0.7, color=color)
            
            # Add count labels on bars
            for j, (bar, tp) in enumerate(zip(bars, timepoints_to_show)):
                model_tp_data = model_data[model_data['timepoint'] == tp]
                if len(model_tp_data) > 0 and bar.get_height() > 0.1:
                    count = model_tp_data['best_count'].iloc[0]
                    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                           f'{count}', ha='center', va='bottom', fontsize=7)
        
        ax4.set_xlabel('Timepoint')
        ax4.set_ylabel('Enrichment in Best Performers')
        ax4.set_title('Timepoint Enrichment: Best Performers')
        ax4.set_xticks(x + width/2)
        ax4.set_xticklabels(timepoints_to_show, rotation=45)
        ax4.axhline(y=1, color='black', linestyle='--', alpha=0.5)
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    # Replace the bottom two panels (axes 5 and 6) with this corrected code:

    # 5. RANDOM INIT: CELL TYPE ANALYSIS (Bottom Left)
    ax5 = axes[2, 0]

    if 'Random Init' in aggregated_data:
        random_data = aggregated_data['Random Init']
        combined_df = random_data['combined_df']
        poorest = random_data['poorest']
        best = random_data['best']
        
        # Get ALL cell types that appear in either poor or best performers
        all_celltypes = set(poorest['zebrafish_anatomy_ontology_class_fine'].unique()) | \
                        set(best['zebrafish_anatomy_ontology_class_fine'].unique())
        
        celltype_analysis = []
        for celltype in all_celltypes:
            total_count = (combined_df['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            poor_count = (poorest['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            best_count = (best['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            
            if total_count > 0:
                failure_rate = (poor_count / total_count) * 100
                success_rate = (best_count / total_count) * 100
                
                celltype_analysis.append({
                    'celltype': celltype,
                    'failure_rate': failure_rate,
                    'success_rate': success_rate,
                    'poor_count': poor_count,
                    'best_count': best_count,
                    'total_count': total_count
                })
        
        if celltype_analysis:
            celltype_df = pd.DataFrame(celltype_analysis)
            # Sort by total occurrence count (most frequent cell types first)
            celltype_df = celltype_df.sort_values('total_count', ascending=True).tail(10)
            
            # Create horizontal bar plot
            y_pos = np.arange(len(celltype_df))
            
            # Plot failure rates (red, extending right)
            bars1 = ax5.barh(y_pos, celltype_df['failure_rate'], alpha=0.7, 
                        color='red', label='Failure Rate', height=0.4)
            
            # Plot success rates (green, extending left from 0)
            bars2 = ax5.barh(y_pos, -celltype_df['success_rate'], alpha=0.7, 
                        color='green', label='Success Rate', height=0.4)
            
            ax5.set_yticks(y_pos)
            ax5.set_yticklabels([f"{ct[:15]}... (n={row['total_count']})" if len(ct) > 15 
                            else f"{ct} (n={row['total_count']})" 
                            for ct, (_, row) in zip(celltype_df['celltype'], celltype_df.iterrows())], 
                            fontsize=9)
            ax5.set_xlabel('← Success Rate (%)    |    Failure Rate (%) →')
            ax5.set_title('Random Init: Cell Type Performance\n(Sorted by Occurrence Count)')
            ax5.axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
            ax5.legend(loc='upper right')
            ax5.grid(True, alpha=0.3)
            
            # Add count labels
            for i, row in enumerate(celltype_df.itertuples()):
                # Failure rate label (right side)
                if row.failure_rate > 2:
                    ax5.text(row.failure_rate + 1, i, 
                        f'{row.failure_rate:.1f}%\n({row.poor_count})', 
                        ha='left', va='center', fontsize=7, color='darkred')
                
                # Success rate label (left side)
                if row.success_rate > 2:
                    ax5.text(-row.success_rate - 1, i, 
                        f'{row.success_rate:.1f}%\n({row.best_count})', 
                        ha='right', va='center', fontsize=7, color='darkgreen')

    # 6. HUMAN BORZOI: CELL TYPE ANALYSIS (Bottom Right)
    ax6 = axes[2, 1]

    if 'Human Borzoi' in aggregated_data:
        borzoi_data = aggregated_data['Human Borzoi']
        combined_df = borzoi_data['combined_df']
        poorest = borzoi_data['poorest']
        best = borzoi_data['best']
        
        # Get ALL cell types that appear in either poor or best performers
        all_celltypes = set(poorest['zebrafish_anatomy_ontology_class_fine'].unique()) | \
                        set(best['zebrafish_anatomy_ontology_class_fine'].unique())
        
        celltype_analysis = []
        for celltype in all_celltypes:
            total_count = (combined_df['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            poor_count = (poorest['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            best_count = (best['zebrafish_anatomy_ontology_class_fine'] == celltype).sum()
            
            if total_count > 0:
                failure_rate = (poor_count / total_count) * 100
                success_rate = (best_count / total_count) * 100
                
                celltype_analysis.append({
                    'celltype': celltype,
                    'failure_rate': failure_rate,
                    'success_rate': success_rate,
                    'poor_count': poor_count,
                    'best_count': best_count,
                    'total_count': total_count
                })
        
        if celltype_analysis:
            celltype_df = pd.DataFrame(celltype_analysis)
            # Sort by total occurrence count (most frequent cell types first)
            celltype_df = celltype_df.sort_values('total_count', ascending=True).tail(10)
            
            # Create horizontal bar plot
            y_pos = np.arange(len(celltype_df))
            
            # Plot failure rates (red, extending right)
            bars1 = ax6.barh(y_pos, celltype_df['failure_rate'], alpha=0.7, 
                        color='red', label='Failure Rate', height=0.4)
            
            # Plot success rates (green, extending left from 0)
            bars2 = ax6.barh(y_pos, -celltype_df['success_rate'], alpha=0.7, 
                        color='green', label='Success Rate', height=0.4)
            
            ax6.set_yticks(y_pos)
            ax6.set_yticklabels([f"{ct[:15]}... (n={row['total_count']})" if len(ct) > 15 
                            else f"{ct} (n={row['total_count']})" 
                            for ct, (_, row) in zip(celltype_df['celltype'], celltype_df.iterrows())], 
                            fontsize=9)
            ax6.set_xlabel('← Success Rate (%)    |    Failure Rate (%) →')
            ax6.set_title('Human Borzoi: Cell Type Performance\n(Sorted by Occurrence Count)')
            ax6.axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
            ax6.legend(loc='upper right')
            ax6.grid(True, alpha=0.3)
            
            # Add count labels
            for i, row in enumerate(celltype_df.itertuples()):
                # Failure rate label (right side)
                if row.failure_rate > 2:
                    ax6.text(row.failure_rate + 1, i, 
                        f'{row.failure_rate:.1f}%\n({row.poor_count})', 
                        ha='left', va='center', fontsize=7, color='darkred')
                
                # Success rate label (left side)
                if row.success_rate > 2:
                    ax6.text(-row.success_rate - 1, i, 
                        f'{row.success_rate:.1f}%\n({row.best_count})', 
                        ha='right', va='center', fontsize=7, color='darkgreen')
    
    plt.tight_layout()
    plt.show()
    
    # Print comprehensive summary
    print("\n" + "="*80)
    print("MODEL COMPARISON SUMMARY")
    print("="*80)
    
    for model_type, data in aggregated_data.items():
        combined_df = data['combined_df']
        poorest = data['poorest']
        best = data['best']
        
        print(f"\n🔬 {model_type.upper()}:")
        print(f"   • Replicates: {len(data['replicates'])}")
        print(f"   • Total pseudobulks: {len(combined_df):,}")
        print(f"   • Mean performance: {combined_df['test_pearson'].mean():.3f} ± {combined_df['test_pearson'].std():.3f}")
        print(f"   • Poor performers mean counts: {poorest['total_counts'].mean():,.0f}")
        print(f"   • Best performers mean counts: {best['total_counts'].mean():,.0f}")
        print(f"   • Count ratio (poor/best): {poorest['total_counts'].mean() / best['total_counts'].mean():.2f}")
    
    return fig, aggregated_data, model_groups

# Usage with your exact format:
# fig, agg_data, model_groups = load_and_analyze_model_replicates(
#     model_paths, model_names, n_performers=25
# )

In [ ]:
# Your data
model_paths = [
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/0/task_0/lr_1.154578199918017e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-0_20250529_Random_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/5/task_5/lr_7.116198906552475e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-5_20250529_Random_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/1/task_1/lr_1.534060288605152e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-1_20250529_Random_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/2/task_2/lr_2.9680669393835383e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-2_20250529_Random_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/3/task_3/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-3_20250529_Human_Borzoi_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/2/task_2/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-2_20250529_Human_Borzoi_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/1/task_1/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-1_20250529_Human_Borzoi_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-0_20250529_Human_Borzoi_0.h5ad"
]

model_names = [
    "Random_0", "Random_1", "Random_2", "Random_3",
    "Human_Borzoi_3", "Human_Borzoi_2", "Human_Borzoi_1", "Human_Borzoi_0"
]

# Run the analysis
fig, aggregated_data, model_groups = load_and_analyze_model_replicates(
    model_paths, model_names, n_performers=50
)

In [ ]:
# Your data
model_paths = [
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed42/version_0/data_out_decima_random_lr3e-06_seed42_20250619_Random_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed43/version_0/data_out_decima_random_lr3e-06_seed43_20250619_Random_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed44/version_0/data_out_decima_random_lr3e-06_seed44_20250619_Random_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed45/version_0/data_out_decima_random_lr3e-06_seed45_20250619_Random_3.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep0_lr3e-05_seed42_20250619_Human_Borzoi_0.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep1_lr3e-05_seed42_20250619_Human_Borzoi_1.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep2_lr3e-05_seed42_20250619_Human_Borzoi_2.h5ad",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep3_lr3e-05_seed42_20250619_Human_Borzoi_3.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep0_lr3e-05_seed42_20250619_Mouse_Borzoi_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep1_lr3e-05_seed42_20250619_Mouse_Borzoi_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep2_lr3e-05_seed42_20250619_Mouse_Borzoi_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep3_lr3e-05_seed42_20250619_Mouse_Borzoi_3.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep0_lr3e-05_seed42_20250619_Human_Decima_0.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep1_lr3e-05_seed42_20250619_Human_Decima_1.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep2_lr3e-05_seed42_20250619_Human_Decima_2.h5ad",
    # "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep3_lr3e-05_seed42_20250619_Human_Decima_3.h5ad"
]

model_names = [
    "Random_0", "Random_1", "Random_2", "Random_3",
    "Human_Borzoi_0", "Human_Borzoi_1", "Human_Borzoi_2", "Human_Borzoi_3"
]

# Run the analysis
fig, aggregated_data, model_groups = load_and_analyze_model_replicates(
    model_paths, model_names, n_performers=50
)

### Figure 4 of DanioDecima MS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

def analyze_timepoint_enrichment_all_models(model_paths, model_names, n_performers=50, figsize=(16, 8)):
    """
    Focused analysis of timepoint enrichment patterns across all four model types
    Shows poor vs best performer enrichment for developmental timepoints
    
    Parameters:
    -----------
    model_paths : list
        List of file paths to model results  
    model_names : list
        List of corresponding model names
    n_performers : int
        Number of poorest and best performers to analyze per model
    figsize : tuple
        Figure size for publication quality
    """
    
    print("="*80)
    print("TIMEPOINT ENRICHMENT ANALYSIS - ALL MODELS")
    print("="*80)
    
    # Define model type mapping with cleaner names
    model_type_mapping = {
        'Random': 'Random Init',
        'Human_Borzoi': 'Human-Borzoi', 
        'Mouse_Borzoi': 'Mouse-Borzoi',
        'Human_Decima': 'Human-Decima'
    }
    
    # Group models by type
    model_groups = {}
    
    for path, name in zip(model_paths, model_names):
        # Extract model type from name
        model_type = None
        for key in model_type_mapping.keys():
            if key in name:
                model_type = model_type_mapping[key]
                break
        
        if model_type is None:
            print(f"⚠️  Could not determine model type for {name}")
            continue
            
        if model_type not in model_groups:
            model_groups[model_type] = []
        
        print(f"Loading {name} as {model_type}...")
        try:
            ad = sc.read_h5ad(path)
            model_groups[model_type].append((ad, name))
            print(f"  ✓ Loaded: {ad.n_obs} pseudobulks")
        except Exception as e:
            print(f"  ✗ Failed to load {name}: {e}")
    
    print(f"\nFound model groups: {list(model_groups.keys())}")
    for group, models in model_groups.items():
        print(f"  • {group}: {len(models)} replicates")
    
    # Aggregate data for each model type
    aggregated_data = {}
    
    for model_type, replicate_list in model_groups.items():
        print(f"\n📊 Processing {model_type}: {len(replicate_list)} replicates")
        
        # Combine all replicates for this model type
        combined_obs_list = []
        for ad, name in replicate_list:
            df_rep = ad.obs.copy().dropna(subset=['test_pearson'])
            df_rep['replicate'] = name
            df_rep['model_type'] = model_type
            combined_obs_list.append(df_rep)
        
        if not combined_obs_list:
            print(f"  ⚠️  No valid data for {model_type}")
            continue
            
        combined_df = pd.concat(combined_obs_list, ignore_index=True)
        
        # Get poor and best performers for this model (across all replicates)
        poorest_model = combined_df.nsmallest(n_performers, 'test_pearson').copy()
        best_model = combined_df.nlargest(n_performers, 'test_pearson').copy()
        
        aggregated_data[model_type] = {
            'combined_df': combined_df,
            'poorest': poorest_model,
            'best': best_model,
            'n_replicates': len(replicate_list)
        }
        
        print(f"   • Total pseudobulks: {len(combined_df):,}")
        print(f"   • Mean performance: {combined_df['test_pearson'].mean():.3f} ± {combined_df['test_pearson'].std():.3f}")
        print(f"   • Range: {combined_df['test_pearson'].min():.3f} - {combined_df['test_pearson'].max():.3f}")
    
    # Define developmental timepoint order
    timepoint_order = ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf', '24hpf', '2dpf', '3dpf', '5dpf', '10dpf']
    
    # Define colors for each model type
    model_colors = {
        'Random Init': '#d62728',      # Red
        'Mouse-Borzoi': '#ff7f0e',     # Orange
        'Human-Borzoi': '#2ca02c',     # Green  
        'Human-Decima': '#1f77b4'      # Blue
    }
    
    # Create publication-quality figure with 2 panels and proper spacing
    fig = plt.figure(figsize=figsize)
    
    # Create subplots with more space for titles
    gs = fig.add_gridspec(1, 2, hspace=0.3, wspace=0.25, top=0.85, bottom=0.15, left=0.08, right=0.92)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Main title with proper spacing
    fig.suptitle('Developmental Timepoint Enrichment in Model Performance Extremes', 
                 fontsize=16, fontweight='bold', y=0.95)
    
    # Panel 1: Poor Performers Enrichment
    print("\n📈 Calculating poor performer enrichment...")
    
    poor_timepoint_data = []
    model_types = list(aggregated_data.keys())
    
    for model_type, data in aggregated_data.items():
        combined_df = data['combined_df']
        poorest = data['poorest']
        
        existing_timepoints = [tp for tp in timepoint_order if tp in combined_df['timepoint'].unique()]
        
        for timepoint in existing_timepoints:
            poor_count = (poorest['timepoint'] == timepoint).sum()
            total_poor = len(poorest)
            overall_count = (combined_df['timepoint'] == timepoint).sum()
            total_overall = len(combined_df)
            
            poor_pct = (poor_count / total_poor) * 100 if total_poor > 0 else 0
            overall_pct = (overall_count / total_overall) * 100 if total_overall > 0 else 0
            
            enrichment = poor_pct / overall_pct if overall_pct > 0 else 0
            
            poor_timepoint_data.append({
                'model': model_type,
                'timepoint': timepoint,
                'poor_count': poor_count,
                'total_poor': total_poor,
                'overall_count': overall_count,
                'total_overall': total_overall,
                'poor_pct': poor_pct,
                'overall_pct': overall_pct,
                'enrichment': enrichment
            })
    
    if poor_timepoint_data:
        poor_df = pd.DataFrame(poor_timepoint_data)
        
        # Get all timepoints that appear in the data
        timepoints_present = [tp for tp in timepoint_order if tp in poor_df['timepoint'].unique()]
        x = np.arange(len(timepoints_present))
        width = 0.2  # Narrower bars for 4 models
        
        # Calculate max enrichment for setting y-axis limit
        max_poor_enrichment = poor_df['enrichment'].max()
        
        # Plot bars for each model
        for i, model_type in enumerate(sorted(model_types)):
            if model_type in poor_df['model'].values:
                model_data = poor_df[poor_df['model'] == model_type]
                enrichments = []
                counts = []
                
                for tp in timepoints_present:
                    tp_data = model_data[model_data['timepoint'] == tp]
                    if len(tp_data) > 0:
                        enrichments.append(tp_data['enrichment'].iloc[0])
                        counts.append(tp_data['poor_count'].iloc[0])
                    else:
                        enrichments.append(0)
                        counts.append(0)
                
                bars = ax1.bar(x + i*width, enrichments, width, 
                             label=f'{model_type} (n={aggregated_data[model_type]["n_replicates"]})', 
                             alpha=0.8, color=model_colors.get(model_type, 'gray'))
                
                # Add count labels on bars (only if enrichment > 0.3 and count > 0)
                for j, (bar, count, enrich) in enumerate(zip(bars, counts, enrichments)):
                    if enrich > 0.3 and count > 0:
                        # Position label at 90% of bar height to avoid crowding
                        label_y = bar.get_height() * 0.9 if bar.get_height() > 0.5 else bar.get_height() + 0.05
                        ax1.text(bar.get_x() + bar.get_width()/2, label_y, 
                               f'{count}', ha='center', va='bottom' if bar.get_height() <= 0.5 else 'center', 
                               fontsize=8, fontweight='bold', color='white' if bar.get_height() > 0.5 else 'black')
        
        ax1.set_xlabel('Developmental Timepoint', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Enrichment in Poor Performers', fontsize=12, fontweight='bold')
        ax1.set_title('A. Poor Performer Timepoint Enrichment', fontsize=13, fontweight='bold', pad=10)
        ax1.set_xticks(x + width*1.5)  # Center the x-tick labels
        ax1.set_xticklabels(timepoints_present, rotation=45, ha='right', fontsize=10)
        ax1.axhline(y=1, color='black', linestyle='--', alpha=0.7, linewidth=1)
        ax1.text(0.02, 0.98, 'Expected level', transform=ax1.transAxes, 
                va='top', ha='left', fontsize=9, alpha=0.7)
        ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Set reasonable y-axis limit (cap at 4.0 to prevent extreme bars)
        y_max = min(max_poor_enrichment * 1.1, 4.0)
        ax1.set_ylim(0, y_max)
    
    # Panel 2: Best Performers Enrichment  
    print("📈 Calculating best performer enrichment...")
    
    best_timepoint_data = []
    
    for model_type, data in aggregated_data.items():
        combined_df = data['combined_df']
        best = data['best']
        
        existing_timepoints = [tp for tp in timepoint_order if tp in combined_df['timepoint'].unique()]
        
        for timepoint in existing_timepoints:
            best_count = (best['timepoint'] == timepoint).sum()
            total_best = len(best)
            overall_count = (combined_df['timepoint'] == timepoint).sum()
            total_overall = len(combined_df)
            
            best_pct = (best_count / total_best) * 100 if total_best > 0 else 0
            overall_pct = (overall_count / total_overall) * 100 if total_overall > 0 else 0
            
            enrichment = best_pct / overall_pct if overall_pct > 0 else 0
            
            best_timepoint_data.append({
                'model': model_type,
                'timepoint': timepoint,
                'best_count': best_count,
                'total_best': total_best,
                'overall_count': overall_count,
                'total_overall': total_overall,
                'best_pct': best_pct,
                'overall_pct': overall_pct,
                'enrichment': enrichment
            })
    
    if best_timepoint_data:
        best_df = pd.DataFrame(best_timepoint_data)
        
        # Calculate max enrichment for setting y-axis limit
        max_best_enrichment = best_df['enrichment'].max()
        
        # Plot bars for each model
        for i, model_type in enumerate(sorted(model_types)):
            if model_type in best_df['model'].values:
                model_data = best_df[best_df['model'] == model_type]
                enrichments = []
                counts = []
                
                for tp in timepoints_present:
                    tp_data = model_data[model_data['timepoint'] == tp]
                    if len(tp_data) > 0:
                        enrichments.append(tp_data['enrichment'].iloc[0])
                        counts.append(tp_data['best_count'].iloc[0])
                    else:
                        enrichments.append(0)
                        counts.append(0)
                
                bars = ax2.bar(x + i*width, enrichments, width, 
                             label=f'{model_type} (n={aggregated_data[model_type]["n_replicates"]})', 
                             alpha=0.8, color=model_colors.get(model_type, 'gray'))
                
                # Add count labels on bars (only if enrichment > 0.3 and count > 0)
                for j, (bar, count, enrich) in enumerate(zip(bars, counts, enrichments)):
                    if enrich > 0.3 and count > 0:
                        # Position label at 90% of bar height to avoid crowding
                        label_y = bar.get_height() * 0.9 if bar.get_height() > 0.5 else bar.get_height() + 0.05
                        ax2.text(bar.get_x() + bar.get_width()/2, label_y, 
                               f'{count}', ha='center', va='bottom' if bar.get_height() <= 0.5 else 'center', 
                               fontsize=8, fontweight='bold', color='white' if bar.get_height() > 0.5 else 'black')
        
        ax2.set_xlabel('Developmental Timepoint', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Enrichment in Best Performers', fontsize=12, fontweight='bold')
        ax2.set_title('B. Best Performer Timepoint Enrichment', fontsize=13, fontweight='bold', pad=10)
        ax2.set_xticks(x + width*1.5)  # Center the x-tick labels
        ax2.set_xticklabels(timepoints_present, rotation=45, ha='right', fontsize=10)
        ax2.axhline(y=1, color='black', linestyle='--', alpha=0.7, linewidth=1)
        ax2.text(0.02, 0.98, 'Expected level', transform=ax2.transAxes, 
                va='top', ha='left', fontsize=9, alpha=0.7)
        ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Set reasonable y-axis limit (cap at 6.0 to prevent extreme bars)
        y_max = min(max_best_enrichment * 1.1, 6.0)
        ax2.set_ylim(0, y_max)
    
    # Print detailed summary
    print("\n" + "="*80)
    print("TIMEPOINT ENRICHMENT SUMMARY")
    print("="*80)
    
    for model_type in sorted(model_types):
        print(f"\n🔬 {model_type.upper()}:")
        data = aggregated_data[model_type]
        print(f"   • Replicates: {data['n_replicates']}")
        print(f"   • Total pseudobulks: {len(data['combined_df']):,}")
        print(f"   • Performance: {data['combined_df']['test_pearson'].mean():.3f} ± {data['combined_df']['test_pearson'].std():.3f}")
        
        # Show top enriched timepoints for poor performers
        if model_type in poor_df['model'].values:
            model_poor = poor_df[poor_df['model'] == model_type].sort_values('enrichment', ascending=False)
            print(f"   • Top poor timepoints: {', '.join([f'{row.timepoint}({row.enrichment:.1f}x)' for _, row in model_poor.head(3).iterrows()])}")
        
        # Show top enriched timepoints for best performers  
        if model_type in best_df['model'].values:
            model_best = best_df[best_df['model'] == model_type].sort_values('enrichment', ascending=False)
            print(f"   • Top best timepoints: {', '.join([f'{row.timepoint}({row.enrichment:.1f}x)' for _, row in model_best.head(3).iterrows()])}")
    
    # Key insights
    print(f"\n🔍 KEY INSIGHTS:")
    print(f"   • Analysis based on {n_performers} poorest/best performers per model type")
    print(f"   • Enrichment > 1.0 indicates over-representation vs expected")
    print(f"   • Later timepoints (2dpf+) show interesting patterns for pretrained models")
    
    return fig, aggregated_data, poor_df, best_df

# Usage with all four model types:
if __name__ == "__main__":
    # Your complete model paths (uncommented)
    model_paths = [
        # Random Init models
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed42/version_0/data_out_decima_random_lr3e-06_seed42_20250619_Random_0.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed43/version_0/data_out_decima_random_lr3e-06_seed43_20250619_Random_1.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed44/version_0/data_out_decima_random_lr3e-06_seed44_20250619_Random_2.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed45/version_0/data_out_decima_random_lr3e-06_seed45_20250619_Random_3.h5ad",
        
        # Human-Borzoi models
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep0_lr3e-05_seed42_20250619_Human_Borzoi_0.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep1_lr3e-05_seed42_20250619_Human_Borzoi_1.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep2_lr3e-05_seed42_20250619_Human_Borzoi_2.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-human_rep3_lr3e-05_seed42_20250619_Human_Borzoi_3.h5ad",
        
        # Mouse-Borzoi models
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep0_lr3e-05_seed42_20250619_Mouse_Borzoi_0.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep1_lr3e-05_seed42_20250619_Mouse_Borzoi_1.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep2_lr3e-05_seed42_20250619_Mouse_Borzoi_2.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_wandb-mouse_rep3_lr3e-05_seed42_20250619_Mouse_Borzoi_3.h5ad",
        
        # Human-Decima models
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep0_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep0_lr3e-05_seed42_20250619_Human_Decima_0.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep1_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep1_lr3e-05_seed42_20250619_Human_Decima_1.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep2_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep2_lr3e-05_seed42_20250619_Human_Decima_2.h5ad",
        "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep3_lr3e-05_seed42/version_0/data_out_decima_pretrained_decima-human_rep3_lr3e-05_seed42_20250619_Human_Decima_3.h5ad"
    ]

    model_names = [
        "Random_0", "Random_1", "Random_2", "Random_3",
        "Human_Borzoi_0", "Human_Borzoi_1", "Human_Borzoi_2", "Human_Borzoi_3",
        "Mouse_Borzoi_0", "Mouse_Borzoi_1", "Mouse_Borzoi_2", "Mouse_Borzoi_3", 
        "Human_Decima_0", "Human_Decima_1", "Human_Decima_2", "Human_Decima_3"
    ]

    # Run the focused timepoint analysis
    fig, agg_data, poor_df, best_df = analyze_timepoint_enrichment_all_models(
        model_paths, model_names, n_performers=50, figsize=(16, 8)
    )
    
    plt.show()

In [ ]:
def analyze_celltype_lift(aggregated_data, figsize=(16, 10)):
    """
    Analyze the lift in Pearson R correlation that Human Borzoi provides over Random Init
    for each cell type
    
    Parameters:
    -----------
    aggregated_data : dict
        Dictionary from the main analysis function containing model data
    figsize : tuple
        Figure size
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    
    print("\n" + "="*80)
    print("CELL TYPE LIFT ANALYSIS: HUMAN BORZOI vs RANDOM INIT")
    print("="*80)
    
    if 'Random Init' not in aggregated_data or 'Human Borzoi' not in aggregated_data:
        print("❌ Both Random Init and Human Borzoi data required for comparison")
        return None
    
    random_data = aggregated_data['Random Init']['combined_df']
    borzoi_data = aggregated_data['Human Borzoi']['combined_df']
    
    # Get all cell types that appear in both models
    random_celltypes = set(random_data['zebrafish_anatomy_ontology_class_fine'].unique())
    borzoi_celltypes = set(borzoi_data['zebrafish_anatomy_ontology_class_fine'].unique())
    common_celltypes = random_celltypes & borzoi_celltypes
    
    print(f"📊 Found {len(common_celltypes)} cell types present in both models")
    
    # Calculate lift for each cell type
    lift_analysis = []
    
    for celltype in common_celltypes:
        # Random Init stats
        random_subset = random_data[random_data['zebrafish_anatomy_ontology_class_fine'] == celltype]
        random_mean_pearson = random_subset['test_pearson'].mean()
        random_std_pearson = random_subset['test_pearson'].std()
        random_count = len(random_subset)
        
        # Human Borzoi stats
        borzoi_subset = borzoi_data[borzoi_data['zebrafish_anatomy_ontology_class_fine'] == celltype]
        borzoi_mean_pearson = borzoi_subset['test_pearson'].mean()
        borzoi_std_pearson = borzoi_subset['test_pearson'].std()
        borzoi_count = len(borzoi_subset)
        
        # Calculate lift and relative improvement
        lift = borzoi_mean_pearson - random_mean_pearson
        relative_improvement = (lift / random_mean_pearson) * 100 if random_mean_pearson > 0 else 0
        
        # Only include cell types with reasonable sample sizes
        if random_count >= 3 and borzoi_count >= 3:
            lift_analysis.append({
                'celltype': celltype,
                'random_mean': random_mean_pearson,
                'random_std': random_std_pearson,
                'random_count': random_count,
                'borzoi_mean': borzoi_mean_pearson,
                'borzoi_std': borzoi_std_pearson,
                'borzoi_count': borzoi_count,
                'lift': lift,
                'relative_improvement': relative_improvement,
                'total_samples': random_count + borzoi_count
            })
    
    if not lift_analysis:
        print("❌ No cell types with sufficient samples for comparison")
        return None
    
    lift_df = pd.DataFrame(lift_analysis)
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Cell Type Performance Lift: Human Borzoi vs Random Init', 
                 fontsize=16, fontweight='bold')
    
    # 1. LIFT BY CELL TYPE (Top Left) - sorted by total samples
    ax1 = axes[0, 0]
    
    # Sort by total samples to prioritize well-represented cell types
    lift_sorted = lift_df.sort_values('total_samples', ascending=True).tail(15)  # Top 15 by sample size
    
    y_pos = np.arange(len(lift_sorted))
    colors = ['darkgreen' if x > 0.01 else 'green' if x > 0.005 else 'gray' if x > -0.005 else 'red' 
              for x in lift_sorted['lift']]
    
    bars = ax1.barh(y_pos, lift_sorted['lift'], color=colors, alpha=0.7)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([f"{ct[:20]}... (n={row['total_samples']})" if len(ct) > 20 
                        else f"{ct} (n={row['total_samples']})" 
                        for ct, (_, row) in zip(lift_sorted['celltype'], lift_sorted.iterrows())], 
                       fontsize=9)
    ax1.set_xlabel('Pearson R Lift (Human Borzoi - Random)')
    ax1.set_title('Performance Lift by Cell Type\n(Sorted by Sample Size)')
    ax1.axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
    ax1.grid(True, alpha=0.3)
    
    # Add lift values on bars
    for i, (bar, row) in enumerate(zip(bars, lift_sorted.itertuples())):
        if abs(bar.get_width()) > 0.003:
            ax1.text(bar.get_width() + (0.002 if bar.get_width() > 0 else -0.005), 
                    bar.get_y() + bar.get_height()/2, 
                    f'{row.lift:+.3f}', ha='left' if bar.get_width() > 0 else 'right', 
                    va='center', fontsize=8, fontweight='bold')
    
    # 2. LIFT DISTRIBUTION (Top Right)
    ax2 = axes[0, 1]
    
    # Histogram of lift values
    ax2.hist(lift_df['lift'], bins=20, alpha=0.7, edgecolor='black', color='skyblue')
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No improvement')
    ax2.axvline(x=lift_df['lift'].mean(), color='green', linestyle='-', linewidth=2, 
               label=f'Mean lift: {lift_df["lift"].mean():.3f}')
    ax2.set_xlabel('Pearson R Lift')
    ax2.set_ylabel('Number of Cell Types')
    ax2.set_title('Distribution of Performance Lift')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add statistics
    n_improved = (lift_df['lift'] > 0).sum()
    n_total = len(lift_df)
    pct_improved = (n_improved / n_total) * 100
    
    ax2.text(0.05, 0.95, f'Cell types improved: {n_improved}/{n_total} ({pct_improved:.1f}%)', 
            transform=ax2.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    
    # 3. SCATTER PLOT: Random vs Borzoi Performance (Bottom Left)
    ax3 = axes[1, 0]
    
    # Size points by total sample count
    sizes = lift_df['total_samples'] * 3  # Scale for visibility
    colors_scatter = ['green' if x > 0 else 'red' for x in lift_df['lift']]
    
    scatter = ax3.scatter(lift_df['random_mean'], lift_df['borzoi_mean'], 
                         s=sizes, c=colors_scatter, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    # Add diagonal line (y=x) for reference
    min_val = min(lift_df['random_mean'].min(), lift_df['borzoi_mean'].min())
    max_val = max(lift_df['random_mean'].max(), lift_df['borzoi_mean'].max())
    ax3.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Equal performance')
    
    ax3.set_xlabel('Random Init Mean Pearson R')
    ax3.set_ylabel('Human Borzoi Mean Pearson R')
    ax3.set_title('Performance Comparison\n(Point size = sample count)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Calculate and display correlation
    from scipy import stats
    corr_r, corr_p = stats.pearsonr(lift_df['random_mean'], lift_df['borzoi_mean'])
    ax3.text(0.05, 0.95, f'Correlation: r={corr_r:.3f} (p={corr_p:.3f})', 
            transform=ax3.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 4. TOP IMPROVEMENTS TABLE (Bottom Right)
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Show top improvements
    top_improvements = lift_df.nlargest(8, 'lift')[['celltype', 'lift', 'relative_improvement', 
                                                    'random_mean', 'borzoi_mean', 'total_samples']]
    
    # Create table data
    table_data = []
    for _, row in top_improvements.iterrows():
        table_data.append([
            row['celltype'][:15] + '...' if len(row['celltype']) > 15 else row['celltype'],
            f"{row['lift']:+.3f}",
            f"{row['relative_improvement']:+.1f}%",
            f"{row['random_mean']:.3f}",
            f"{row['borzoi_mean']:.3f}",
            f"{row['total_samples']}"
        ])
    
    if table_data:
        table = ax4.table(cellText=table_data,
                         colLabels=['Cell Type', 'Lift', 'Rel. Imp.', 'Random', 'Borzoi', 'n'],
                         cellLoc='center',
                         loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.2, 1.5)
        ax4.set_title('Top Improvements', pad=20, fontsize=12, fontweight='bold')
        
        # Color code the improvement cells
        for i in range(len(table_data)):
            # Color the lift column based on improvement
            lift_val = float(table_data[i][1])
            if lift_val > 0.01:
                table[(i+1, 1)].set_facecolor('lightgreen')
            elif lift_val > 0.005:
                table[(i+1, 1)].set_facecolor('lightyellow')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed summary
    print(f"\n📈 LIFT ANALYSIS SUMMARY:")
    print(f"   • Total cell types analyzed: {len(lift_df)}")
    print(f"   • Cell types with improved performance: {n_improved} ({pct_improved:.1f}%)")
    print(f"   • Mean lift across all cell types: {lift_df['lift'].mean():.4f}")
    print(f"   • Median lift: {lift_df['lift'].median():.4f}")
    print(f"   • Standard deviation of lift: {lift_df['lift'].std():.4f}")
    
    print(f"\n🏆 TOP 5 IMPROVEMENTS:")
    top_5 = lift_df.nlargest(5, 'lift')
    for i, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"   {i}. {row['celltype']}: +{row['lift']:.3f} ({row['relative_improvement']:+.1f}%)")
        print(f"      Random: {row['random_mean']:.3f} → Borzoi: {row['borzoi_mean']:.3f} (n={row['total_samples']})")
    
    print(f"\n📉 BOTTOM 3 (if any declines):")
    bottom_3 = lift_df.nsmallest(5, 'lift')
    for i, (_, row) in enumerate(bottom_3.iterrows(), 1):
        print(f"   {i}. {row['celltype']}: {row['lift']:+.3f} ({row['relative_improvement']:+.1f}%)")
        print(f"      Random: {row['random_mean']:.3f} → Borzoi: {row['borzoi_mean']:.3f} (n={row['total_samples']})")
    
    return fig, lift_df

# Usage after running the main analysis:
# fig_lift, lift_results = analyze_celltype_lift(agg_data)

In [ ]:
fig_lift, lift_results = analyze_celltype_lift(aggregated_data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

def comprehensive_celltype_timepoint_analysis_fixed(aggregated_data, min_samples=5, figsize=(18, 12)):
    """
    Comprehensive analysis of cell type × timepoint performance across all 4 model types
    Fixed version with better layout and focused panels
    """
    
    print("\n" + "="*80)
    print("COMPREHENSIVE CELL TYPE × TIMEPOINT ANALYSIS")
    print("="*80)
    
    # Define model order and colors
    model_order = ['Random Init', 'Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']
    model_colors = {
        'Random Init': '#d62728',
        'Mouse-Borzoi': '#ff7f0e', 
        'Human-Borzoi': '#2ca02c',
        'Human-Decima': '#1f77b4'
    }
    
    # Define developmental stages
    developmental_stages = {
        'Early (10-19hpf)': ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf'],
        'Mid (24hpf-2dpf)': ['24hpf', '2dpf'],
        'Late (3-10dpf)': ['3dpf', '5dpf', '10dpf']
    }
    
    # 1. CREATE COMPREHENSIVE PERFORMANCE MATRIX
    print("📊 Creating comprehensive performance matrix...")
    
    performance_data = []
    
    for model_name, model_data in aggregated_data.items():
        if model_name not in model_order:
            continue
            
        df = model_data['combined_df']
        
        for _, row in df.iterrows():
            performance_data.append({
                'model': model_name,
                'celltype': row['zebrafish_anatomy_ontology_class_fine'],
                'timepoint': row['timepoint'],
                'pearson': row['test_pearson'],
                'total_counts': row['total_counts']
            })
    
    perf_df = pd.DataFrame(performance_data)
    
    # Filter for combinations with sufficient samples
    celltype_timepoint_counts = perf_df.groupby(['celltype', 'timepoint']).size()
    valid_combinations = celltype_timepoint_counts[celltype_timepoint_counts >= min_samples].index
    
    print(f"   • Total combinations: {len(celltype_timepoint_counts)}")
    print(f"   • Valid combinations (≥{min_samples} samples): {len(valid_combinations)}")
    
    # 2. CALCULATE PERFORMANCE STATISTICS FOR EACH COMBINATION
    print("📈 Calculating performance statistics...")
    
    stats_data = []
    
    for celltype, timepoint in valid_combinations:
        combo_data = perf_df[(perf_df['celltype'] == celltype) & 
                            (perf_df['timepoint'] == timepoint)]
        
        model_stats = {}
        for model in model_order:
            model_subset = combo_data[combo_data['model'] == model]
            if len(model_subset) > 0:
                model_stats[model] = {
                    'mean': model_subset['pearson'].mean(),
                    'std': model_subset['pearson'].std(),
                    'count': len(model_subset),
                    'median': model_subset['pearson'].median()
                }
        
        # Only include if we have data for at least 3 models including Random
        if len(model_stats) >= 3 and 'Random Init' in model_stats:
            # Calculate improvements over random
            random_mean = model_stats['Random Init']['mean']
            
            combo_result = {
                'celltype': celltype,
                'timepoint': timepoint,
                'random_mean': random_mean,
                'total_samples': sum([s['count'] for s in model_stats.values()])
            }
            
            # Add stats for each model and calculate lifts
            best_model = 'Random Init'
            best_performance = random_mean
            
            for model in model_order:
                if model in model_stats:
                    mean_perf = model_stats[model]['mean']
                    combo_result[f'{model}_mean'] = mean_perf
                    combo_result[f'{model}_count'] = model_stats[model]['count']
                    combo_result[f'{model}_lift'] = mean_perf - random_mean
                    combo_result[f'{model}_rel_imp'] = ((mean_perf - random_mean) / random_mean * 100) if random_mean > 0 else 0
                    
                    if mean_perf > best_performance:
                        best_performance = mean_perf
                        best_model = model
                else:
                    combo_result[f'{model}_mean'] = np.nan
                    combo_result[f'{model}_count'] = 0
                    combo_result[f'{model}_lift'] = np.nan
                    combo_result[f'{model}_rel_imp'] = np.nan
            
            combo_result['best_model'] = best_model
            combo_result['max_improvement'] = best_performance - random_mean
            combo_result['max_rel_improvement'] = ((best_performance - random_mean) / random_mean * 100) if random_mean > 0 else 0
            
            # Assign developmental stage
            stage = 'Unknown'
            for stage_name, timepoints in developmental_stages.items():
                if timepoint in timepoints:
                    stage = stage_name
                    break
            combo_result['dev_stage'] = stage
            
            stats_data.append(combo_result)
    
    stats_df = pd.DataFrame(stats_data)
    
    if stats_df.empty:
        print("❌ No valid combinations found")
        return None
    
    print(f"   • Analyzed {len(stats_df)} cell type × timepoint combinations")
    
    # 3. CREATE FIXED VISUALIZATION (4 panels in 2x2 grid)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3, top=0.92, bottom=0.08, left=0.08, right=0.95)
    
    fig.suptitle('Cell Type × Timepoint Performance Analysis: Lift vs Best Performers', 
                 fontsize=16, fontweight='bold')
    
    # Panel 1: Top Improvements Heatmap (Top Left)
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Get top 12 improvements and create heatmap
    top_improvements = stats_df.nlargest(12, 'max_improvement')
    
    heatmap_data = []
    heatmap_labels = []
    
    for _, row in top_improvements.iterrows():
        combo_label = f"{row['celltype'][:12]}...\n{row['timepoint']}" if len(row['celltype']) > 12 else f"{row['celltype']}\n{row['timepoint']}"
        heatmap_labels.append(combo_label)
        
        row_data = []
        for model in model_order:
            lift = row[f'{model}_lift']
            row_data.append(lift if not np.isnan(lift) else 0)
        heatmap_data.append(row_data)
    
    heatmap_array = np.array(heatmap_data)
    im1 = ax1.imshow(heatmap_array, cmap='RdYlGn', aspect='auto', vmin=-0.05, vmax=0.15)
    ax1.set_xticks(range(len(model_order)))
    ax1.set_xticklabels([m.replace('-', '\n') for m in model_order], rotation=45, ha='right', fontsize=10)
    ax1.set_yticks(range(len(heatmap_labels)))
    ax1.set_yticklabels(heatmap_labels, fontsize=9)
    ax1.set_title('A. Top Cell Type × Timepoint Lifts\n(Improvement over Random)', fontweight='bold', pad=15)
    
    # Add text annotations
    for i in range(len(heatmap_labels)):
        for j in range(len(model_order)):
            value = heatmap_array[i, j]
            if abs(value) > 0.01:
                ax1.text(j, i, f'{value:.3f}', ha='center', va='center', 
                        fontsize=8, fontweight='bold', 
                        color='white' if abs(value) > 0.08 else 'black')
    
    # Add colorbar
    cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.8)
    cbar1.set_label('Lift over Random', rotation=270, labelpad=15)
    
    # Panel 2: Developmental Stage Summary (Top Right)
    ax2 = fig.add_subplot(gs[0, 1])
    
    stage_summary = []
    for stage in developmental_stages.keys():
        stage_data = stats_df[stats_df['dev_stage'] == stage]
        if len(stage_data) > 0:
            for model in ['Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']:
                lift_col = f'{model}_lift'
                if lift_col in stage_data.columns:
                    valid_lifts = stage_data[lift_col].dropna()
                    if len(valid_lifts) > 0:
                        stage_summary.append({
                            'stage': stage,
                            'model': model,
                            'mean_lift': valid_lifts.mean(),
                            'median_lift': valid_lifts.median(),
                            'n_combinations': len(valid_lifts),
                            'pct_improved': (valid_lifts > 0).mean() * 100
                        })
    
    if stage_summary:
        stage_df = pd.DataFrame(stage_summary)
        
        # Create grouped bar plot
        stages = list(developmental_stages.keys())
        x = np.arange(len(stages))
        width = 0.25
        
        pretrained_models = ['Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']
        for i, model in enumerate(pretrained_models):
            model_data = stage_df[stage_df['model'] == model]
            lifts = [model_data[model_data['stage'] == stage]['mean_lift'].iloc[0] 
                    if len(model_data[model_data['stage'] == stage]) > 0 else 0 
                    for stage in stages]
            
            bars = ax2.bar(x + i*width, lifts, width, label=model, 
                          color=model_colors[model], alpha=0.8)
            
            # Add count labels
            for j, (bar, stage) in enumerate(zip(bars, stages)):
                stage_subset = model_data[model_data['stage'] == stage]
                if len(stage_subset) > 0 and bar.get_height() > 0.005:
                    n_combos = stage_subset['n_combinations'].iloc[0]
                    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
                           f'n={n_combos}', ha='center', va='bottom', fontsize=8)
        
        ax2.set_xlabel('Developmental Stage', fontweight='bold')
        ax2.set_ylabel('Mean Lift over Random', fontweight='bold')
        ax2.set_title('B. Performance Lift by Developmental Stage', fontweight='bold', pad=15)
        ax2.set_xticks(x + width)
        ax2.set_xticklabels([s.replace(' ', '\n') for s in stages], fontsize=10)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    # Panel 3: Improvement Distribution by Timepoint (Bottom Left)
    ax3 = fig.add_subplot(gs[1, 0])
    
    timepoint_order = ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf', '24hpf', '2dpf', '3dpf', '5dpf', '10dpf']
    timepoints_present = [tp for tp in timepoint_order if tp in stats_df['timepoint'].unique()]
    
    improvement_by_timepoint = []
    timepoint_labels = []
    
    for tp in timepoints_present:
        tp_data = stats_df[stats_df['timepoint'] == tp]
        if len(tp_data) > 0:
            improvement_by_timepoint.append(tp_data['max_improvement'].values)
            timepoint_labels.append(f"{tp}\n(n={len(tp_data)})")
    
    if improvement_by_timepoint:
        bp = ax3.boxplot(improvement_by_timepoint, labels=timepoint_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
            patch.set_alpha(0.7)
        
        ax3.set_xlabel('Timepoint', fontweight='bold')
        ax3.set_ylabel('Max Improvement over Random', fontweight='bold')
        ax3.set_title('C. Cell Type Lift Distribution by Timepoint\n(Best lift per cell type × timepoint)', 
                     fontweight='bold', pad=15)
        ax3.tick_params(axis='x', rotation=45, labelsize=9)
        ax3.grid(True, alpha=0.3, axis='y')
        ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        
        # Add median values as text
        for i, tp_data in enumerate(improvement_by_timepoint):
            median_val = np.median(tp_data)
            if median_val > 0.01:
                ax3.text(i+1, median_val + 0.01, f'{median_val:.3f}', 
                        ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Panel 4: Top Cell Type Champions Table (Bottom Right)
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('off')
    
    # Top cell types by improvement
    top_celltypes = stats_df.nlargest(10, 'max_improvement')
    
    table_data = []
    for _, row in top_celltypes.iterrows():
        best_model = row['best_model']
        improvement = row['max_improvement']
        rel_improvement = row['max_rel_improvement']
        
        table_data.append([
            row['celltype'][:20] + '...' if len(row['celltype']) > 20 else row['celltype'],
            row['timepoint'],
            row['dev_stage'].split(' ')[0],  # Just the stage name
            best_model.replace('-', '\n'),
            f"{improvement:+.3f}",
            f"{rel_improvement:+.1f}%"
        ])
    
    if table_data:
        table = ax4.table(cellText=table_data,
                         colLabels=['Cell Type', 'Time', 'Stage', 'Best\nModel', 'Lift', 'Rel.\nImp.'],
                         cellLoc='center',
                         loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 2.2)
        ax4.set_title('D. Top Cell Type × Timepoint Champions', 
                     pad=20, fontsize=12, fontweight='bold')
        
        # Color code the best model column
        for i in range(len(table_data)):
            best_model = table_data[i][3].replace('\n', '-')
            if best_model in model_colors:
                table[(i+1, 3)].set_facecolor(model_colors[best_model])
                table[(i+1, 3)].set_alpha(0.3)
    
    # Print analysis of the discrepancy
    print("\n" + "="*80)
    print("ANALYSIS: LIFT vs BEST PERFORMER DISCREPANCY")
    print("="*80)
    
    # Analyze the discrepancy between lift and best performer enrichment
    print("\n🔍 RECONCILING LIFT vs BEST PERFORMER PATTERNS:")
    
    # Get timepoint-wise statistics
    timepoint_analysis = []
    for tp in timepoints_present:
        tp_data = stats_df[stats_df['timepoint'] == tp]
        
        # Count how many cell types show improvement at this timepoint
        n_improved = (tp_data['max_improvement'] > 0).sum()
        total_celltypes = len(tp_data)
        pct_improved = (n_improved / total_celltypes * 100) if total_celltypes > 0 else 0
        
        # Average improvement magnitude
        avg_improvement = tp_data['max_improvement'].mean()
        median_improvement = tp_data['max_improvement'].median()
        
        timepoint_analysis.append({
            'timepoint': tp,
            'n_celltypes': total_celltypes,
            'n_improved': n_improved,
            'pct_improved': pct_improved,
            'avg_improvement': avg_improvement,
            'median_improvement': median_improvement,
            'max_improvement': tp_data['max_improvement'].max()
        })
    
    tp_analysis_df = pd.DataFrame(timepoint_analysis)
    
    print(f"\n📊 TIMEPOINT PATTERNS:")
    print(f"{'Timepoint':<10} {'CellTypes':<10} {'%Improved':<10} {'AvgLift':<10} {'MaxLift':<10}")
    print("-" * 60)
    for _, row in tp_analysis_df.iterrows():
        print(f"{row['timepoint']:<10} {row['n_celltypes']:<10} {row['pct_improved']:<10.1f} "
              f"{row['avg_improvement']:<10.3f} {row['max_improvement']:<10.3f}")
    
    print(f"\n💡 KEY INSIGHTS:")
    print(f"   • Late timepoints (3dpf, 5dpf, 10dpf) show more cell types with improvement")
    print(f"   • BUT: The magnitude of improvements may be smaller per cell type")
    print(f"   • Best performer enrichment = more pseudobulks performing well")
    print(f"   • Lift analysis = which specific cell types benefit most")
    print(f"   • These measure different aspects of model performance!")
    
    return fig, stats_df, tp_analysis_df

# Usage:
fig_fixed, comprehensive_stats, timepoint_patterns = comprehensive_celltype_timepoint_analysis_fixed(agg_data)

### Figure 5 from DanioDecima MS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def analyze_developmental_performance_patterns_fixed(aggregated_data, figsize=(18, 14)):
    """
    Analyze the apparent contradiction between best performers and biggest lifts
    across developmental timepoints - FIXED VERSION
    """
    
    print("="*80)
    print("DEVELOPMENTAL PERFORMANCE PATTERN ANALYSIS")
    print("="*80)
    
    # Define model order and timepoints
    model_order = ['Random Init', 'Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']
    timepoint_order = ['10hpf', '12hpf', '14hpf', '16hpf', '19hpf', '24hpf', '2dpf', '3dpf', '5dpf', '10dpf']
    
    # Define developmental stages
    dev_stages = {
        'Early (10-16hpf)': ['10hpf', '12hpf', '14hpf', '16hpf'],
        'Mid (19hpf-24hpf)': ['19hpf', '24hpf'], 
        'Late (2dpf+)': ['2dpf', '3dpf', '5dpf', '10dpf']
    }
    
    # Colors
    model_colors = {
        'Random Init': '#d62728',
        'Mouse-Borzoi': '#ff7f0e', 
        'Human-Borzoi': '#2ca02c',
        'Human-Decima': '#1f77b4'
    }
    
    # 1. COLLECT PERFORMANCE DATA BY TIMEPOINT
    timepoint_stats = []
    
    for timepoint in timepoint_order:
        for model_name, model_data in aggregated_data.items():
            if model_name not in model_order:
                continue
                
            df = model_data['combined_df']
            tp_data = df[df['timepoint'] == timepoint]['test_pearson'].dropna()
            
            if len(tp_data) > 0:
                # Assign developmental stage
                stage = 'Unknown'
                for stage_name, tps in dev_stages.items():
                    if timepoint in tps:
                        stage = stage_name
                        break
                
                timepoint_stats.append({
                    'timepoint': timepoint,
                    'model': model_name,
                    'stage': stage,
                    'mean_pearson': tp_data.mean(),
                    'median_pearson': tp_data.median(),
                    'std_pearson': tp_data.std(),
                    'q75_pearson': tp_data.quantile(0.75),
                    'q90_pearson': tp_data.quantile(0.90),
                    'q95_pearson': tp_data.quantile(0.95),
                    'max_pearson': tp_data.max(),
                    'n_samples': len(tp_data)
                })
    
    tp_stats_df = pd.DataFrame(timepoint_stats)
    
    # 2. CALCULATE LIFTS
    lift_stats = []
    
    for timepoint in timepoint_order:
        tp_subset = tp_stats_df[tp_stats_df['timepoint'] == timepoint]
        random_row = tp_subset[tp_subset['model'] == 'Random Init']
        
        if len(random_row) > 0:
            random_mean = random_row['mean_pearson'].iloc[0]
            random_q75 = random_row['q75_pearson'].iloc[0]
            random_q95 = random_row['q95_pearson'].iloc[0]
            stage = random_row['stage'].iloc[0]
            
            for model in ['Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']:
                model_row = tp_subset[tp_subset['model'] == model]
                if len(model_row) > 0:
                    model_mean = model_row['mean_pearson'].iloc[0]
                    model_q75 = model_row['q75_pearson'].iloc[0]
                    model_q95 = model_row['q95_pearson'].iloc[0]
                    
                    lift_stats.append({
                        'timepoint': timepoint,
                        'model': model,
                        'stage': stage,
                        'mean_lift': model_mean - random_mean,
                        'q75_lift': model_q75 - random_q75,
                        'q95_lift': model_q95 - random_q95,
                        'relative_lift': ((model_mean - random_mean) / random_mean * 100) if random_mean > 0 else 0,
                        'random_baseline': random_mean,
                        'pretrained_performance': model_mean
                    })
    
    lift_df = pd.DataFrame(lift_stats)
    
    # 3. CREATE FIXED VISUALIZATION WITH PROPER SPACING
    fig = plt.figure(figsize=figsize)
    
    # Use gridspec with more spacing and better proportions
    gs = fig.add_gridspec(3, 2, 
                         height_ratios=[1, 1, 0.6],  # Make bottom panel smaller
                         hspace=0.45,  # More vertical spacing
                         wspace=0.25,  # Horizontal spacing
                         top=0.92,     # Leave more room at top
                         bottom=0.08,
                         left=0.08, 
                         right=0.95)
    
    fig.suptitle('Developmental Performance Patterns: Absolute Performance vs Improvement Analysis', 
                 fontsize=16, fontweight='bold', y=0.96)
    
    # Panel 1: Absolute Performance by Timepoint (Top Left)
    ax1 = fig.add_subplot(gs[0, 0])
    
    timepoints_present = [tp for tp in timepoint_order if tp in tp_stats_df['timepoint'].unique()]
    x = np.arange(len(timepoints_present))
    width = 0.2
    
    for i, model in enumerate(model_order):
        model_data = tp_stats_df[tp_stats_df['model'] == model]
        means = [model_data[model_data['timepoint'] == tp]['mean_pearson'].iloc[0] 
                if len(model_data[model_data['timepoint'] == tp]) > 0 else 0 
                for tp in timepoints_present]
        
        bars = ax1.bar(x + i*width, means, width, label=model, 
                      color=model_colors[model], alpha=0.8)
        
        # Highlight peak performance for each model
        if means:
            max_idx = np.argmax(means)
            max_val = max(means)
            if max_val > 0.7:  # Only annotate high peaks
                ax1.text(x[max_idx] + i*width, max_val + 0.02, f'PEAK\n{max_val:.3f}', 
                        ha='center', va='bottom', fontsize=7, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=model_colors[model], alpha=0.3))
    
    ax1.set_xlabel('Developmental Timepoint', fontweight='bold')
    ax1.set_ylabel('Mean Pearson Correlation', fontweight='bold')
    ax1.set_title('A. Absolute Performance by Timepoint', fontweight='bold', pad=15)
    ax1.set_xticks(x + width*1.5)
    ax1.set_xticklabels(timepoints_present, rotation=45, ha='right')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.set_ylim(0, 0.9)
    
    # Panel 2: Lift Analysis (Top Right)
    ax2 = fig.add_subplot(gs[0, 1])
    
    for i, model in enumerate(['Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']):
        model_data = lift_df[lift_df['model'] == model]
        lifts = [model_data[model_data['timepoint'] == tp]['mean_lift'].iloc[0] 
                if len(model_data[model_data['timepoint'] == tp]) > 0 else 0 
                for tp in timepoints_present]
        
        bars = ax2.bar(x + i*width, lifts, width, label=model, 
                      color=model_colors[model], alpha=0.8)
        
        # Highlight biggest lift for each model
        if lifts and max(lifts) > 0:
            max_idx = np.argmax(lifts)
            max_val = max(lifts)
            if max_val > 0.15:  # Only annotate significant lifts
                ax2.text(x[max_idx] + i*width, max_val + 0.01, f'MAX\n{max_val:.3f}', 
                        ha='center', va='bottom', fontsize=7, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=model_colors[model], alpha=0.3))
    
    ax2.set_xlabel('Developmental Timepoint', fontweight='bold')
    ax2.set_ylabel('Lift over Random', fontweight='bold')
    ax2.set_title('B. Improvement over Random by Timepoint', fontweight='bold', pad=15)
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(timepoints_present, rotation=45, ha='right')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Panel 3: Random Baseline Performance (Middle Left)
    ax3 = fig.add_subplot(gs[1, 0])
    
    random_data = tp_stats_df[tp_stats_df['model'] == 'Random Init']
    random_means = [random_data[random_data['timepoint'] == tp]['mean_pearson'].iloc[0] 
                   if len(random_data[random_data['timepoint'] == tp]) > 0 else 0 
                   for tp in timepoints_present]
    
    bars = ax3.bar(timepoints_present, random_means, color='red', alpha=0.7)
    ax3.set_xlabel('Developmental Timepoint', fontweight='bold')
    ax3.set_ylabel('Random Model Performance', fontweight='bold')
    ax3.set_title('C. Random Model Baseline Performance', fontweight='bold', pad=15)
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add text annotations for room for improvement
    for i, (tp, baseline) in enumerate(zip(timepoints_present, random_means)):
        room_for_improvement = 1.0 - baseline  # Assuming perfect correlation = 1.0
        if room_for_improvement > 0.3:  # Only annotate significant room
            ax3.text(i, baseline + 0.02, f'Room:\n{room_for_improvement:.3f}', 
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Panel 4: Lift vs Baseline Scatter (Middle Right)
    ax4 = fig.add_subplot(gs[1, 1])
    
    # Create scatter plot of baseline performance vs lift
    for model in ['Mouse-Borzoi', 'Human-Borzoi', 'Human-Decima']:
        model_data = lift_df[lift_df['model'] == model]
        
        scatter = ax4.scatter(model_data['random_baseline'], model_data['mean_lift'], 
                             label=model, color=model_colors[model], alpha=0.7, s=80)
        
        # Add timepoint labels for interesting points
        for _, row in model_data.iterrows():
            if row['mean_lift'] > 0.15 or row['random_baseline'] < 0.6:  # Only label interesting points
                ax4.annotate(row['timepoint'], 
                            (row['random_baseline'], row['mean_lift']),
                            xytext=(3, 3), textcoords='offset points', 
                            fontsize=8, alpha=0.8, fontweight='bold')
    
    ax4.set_xlabel('Random Model Baseline Performance', fontweight='bold')
    ax4.set_ylabel('Lift over Random', fontweight='bold')
    ax4.set_title('D. Baseline vs Improvement Relationship', fontweight='bold', pad=15)
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Add correlation line
    from scipy import stats
    all_baselines = lift_df['random_baseline'].values
    all_lifts = lift_df['mean_lift'].values
    slope, intercept, r_value, p_value, std_err = stats.linregress(all_baselines, all_lifts)
    line_x = np.linspace(all_baselines.min(), all_baselines.max(), 100)
    line_y = slope * line_x + intercept
    ax4.plot(line_x, line_y, 'k--', alpha=0.5, linewidth=2)
    
    # Add correlation text
    ax4.text(0.05, 0.95, f'Correlation: r={r_value:.3f}\np={p_value:.3f}', 
            transform=ax4.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=10)
    
    # Panel 5: Developmental Stage Summary (Bottom - spans both columns)
    ax5 = fig.add_subplot(gs[2, :])
    
    # Create summary table
    stage_summary = []
    
    for stage_name, stage_tps in dev_stages.items():
        stage_data = {
            'stage': stage_name,
            'timepoints': ', '.join(stage_tps)
        }
        
        # Random baseline for this stage
        stage_random = tp_stats_df[(tp_stats_df['timepoint'].isin(stage_tps)) & 
                                  (tp_stats_df['model'] == 'Random Init')]
        if len(stage_random) > 0:
            stage_data['random_mean'] = stage_random['mean_pearson'].mean()
            stage_data['random_best'] = stage_random['mean_pearson'].max()
        
        # Best pretrained performance for this stage
        stage_pretrained = tp_stats_df[(tp_stats_df['timepoint'].isin(stage_tps)) & 
                                      (tp_stats_df['model'] != 'Random Init')]
        if len(stage_pretrained) > 0:
            stage_data['pretrained_best'] = stage_pretrained['mean_pearson'].max()
            best_model = stage_pretrained.loc[stage_pretrained['mean_pearson'].idxmax(), 'model']
            stage_data['best_model'] = best_model
        
        # Average lift for this stage
        stage_lifts = lift_df[lift_df['timepoint'].isin(stage_tps)]
        if len(stage_lifts) > 0:
            stage_data['avg_lift'] = stage_lifts['mean_lift'].mean()
            stage_data['max_lift'] = stage_lifts['mean_lift'].max()
        
        stage_summary.append(stage_data)
    
    # Create table with better formatting
    table_data = []
    for stage_info in stage_summary:
        table_data.append([
            stage_info['stage'],
            f"{stage_info.get('random_mean', 0):.3f}",
            f"{stage_info.get('pretrained_best', 0):.3f}",
            f"{stage_info.get('avg_lift', 0):.3f}",
            f"{stage_info.get('max_lift', 0):.3f}",
            stage_info.get('best_model', 'N/A')
        ])
    
    table = ax5.table(cellText=table_data,
                     colLabels=['Developmental Stage', 'Random\nBaseline', 'Pretrained\nBest', 
                               'Average\nLift', 'Maximum\nLift', 'Best\nModel'],
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.0, 2.5)
    ax5.set_title('E. Developmental Stage Summary: Performance vs Improvement Patterns', 
                 fontweight='bold', fontsize=14, pad=15)
    ax5.axis('off')
    
    # Color code the table
    for i in range(len(table_data)):
        best_model = table_data[i][5]
        if best_model in model_colors:
            table[(i+1, 5)].set_facecolor(model_colors[best_model])
            table[(i+1, 5)].set_alpha(0.3)
    
    # Print detailed analysis
    print("\n🔍 KEY INSIGHTS FROM THE ANALYSIS:")
    print("="*60)
    
    # Find peaks for each metric
    random_peak_data = tp_stats_df[tp_stats_df['model'] == 'Random Init']
    random_peak_tp = random_peak_data.loc[random_peak_data['mean_pearson'].idxmax(), 'timepoint']
    random_peak_val = random_peak_data['mean_pearson'].max()
    
    pretrained_peak_data = tp_stats_df[tp_stats_df['model'] != 'Random Init']
    pretrained_peak_tp = pretrained_peak_data.loc[pretrained_peak_data['mean_pearson'].idxmax(), 'timepoint']
    pretrained_peak_val = pretrained_peak_data['mean_pearson'].max()
    pretrained_peak_model = pretrained_peak_data.loc[pretrained_peak_data['mean_pearson'].idxmax(), 'model']
    
    max_lift_data = lift_df.loc[lift_df['mean_lift'].idxmax()]
    max_lift_tp = max_lift_data['timepoint']
    max_lift_val = max_lift_data['mean_lift']
    max_lift_model = max_lift_data['model']
    
    print(f"📊 PERFORMANCE PEAKS:")
    print(f"   • Random model peaks at: {random_peak_tp} ({random_peak_val:.3f})")
    print(f"   • Pretrained models peak at: {pretrained_peak_tp} ({pretrained_peak_val:.3f}, {pretrained_peak_model})")
    print(f"   • Biggest lift occurs at: {max_lift_tp} ({max_lift_val:.3f}, {max_lift_model})")
    
    # Test correlation between baseline and lift
    from scipy import stats
    correlation = stats.pearsonr(all_baselines, all_lifts)
    print(f"\n💡 BASELINE vs LIFT RELATIONSHIP:")
    print(f"   • Correlation: r={correlation[0]:.3f}, p={correlation[1]:.3f}")
    
    if correlation[1] < 0.05:  # Significant correlation
        if correlation[0] < -0.3:
            print("   ✓ STRONG NEGATIVE CORRELATION: Lower baseline → Bigger lift")
            print("     This supports the 'room for improvement' hypothesis!")
        elif correlation[0] > 0.3:
            print("   ✓ STRONG POSITIVE CORRELATION: Higher baseline → Bigger lift")
            print("     This suggests pretrained models excel where random models also do well!")
        else:
            print("   ○ WEAK CORRELATION: Mixed pattern")
    else:
        print("   ○ NO SIGNIFICANT CORRELATION: Multiple factors at play")
    
    # Stage-wise insights
    print(f"\n📈 DEVELOPMENTAL STAGE INSIGHTS:")
    for stage_info in stage_summary:
        if 'avg_lift' in stage_info and 'random_mean' in stage_info:
            room = 1.0 - stage_info['random_mean']
            efficiency = stage_info['avg_lift'] / room if room > 0 else 0
            print(f"   • {stage_info['stage']}:")
            print(f"     - Random baseline: {stage_info['random_mean']:.3f}")
            print(f"     - Average lift: {stage_info['avg_lift']:.3f}")
            print(f"     - Room for improvement: {room:.3f}")
            print(f"     - Efficiency (lift/room): {efficiency:.3f}")
    
    return fig, tp_stats_df, lift_df, stage_summary

# Usage:
fig_fixed, tp_stats, lift_stats, stage_analysis = analyze_developmental_performance_patterns_fixed(agg_data)

In [ ]:
def analyze_gene_lift(
    aggregated_data,
    ad_var,  # your ad.var DataFrame, indexed by gene or with a 'gene' column
    gene_col='gene',
    top_n=15,
    figsize=(18, 11)
):
    """
    Analyze the lift in Pearson R correlation that Human Borzoi provides over Random Init
    for each gene, merged with ad.var for gene annotations.

    Parameters
    ----------
    aggregated_data : dict
        As above, output from main analysis.
    ad_var : pandas.DataFrame
        AnnData .var dataframe, indexed by gene ID or name, with gene metadata.
    gene_col : str
        Column in .combined_df containing gene name or ID.
    top_n : int
        How many genes to show in main bar plot (by sample size).
    figsize : tuple
        Figure size.
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

    print("\n" + "="*80)
    print("GENE LIFT ANALYSIS: HUMAN BORZOI vs RANDOM INIT")
    print("="*80)

    if 'Random Init' not in aggregated_data or 'Human Borzoi' not in aggregated_data:
        print("❌ Both Random Init and Human Borzoi data required for comparison")
        return None

    random_data = aggregated_data['Random Init']['combined_df']
    borzoi_data = aggregated_data['Human Borzoi']['combined_df']

    # Ensure gene_col exists in both
    if gene_col not in random_data.columns or gene_col not in borzoi_data.columns:
        print(f"❌ Column '{gene_col}' not found in one or both DataFrames")
        print("Available columns in Random Init:", list(random_data.columns))
        print("Available columns in Human Borzoi:", list(borzoi_data.columns))
        return None

    # Get all genes that appear in both models
    random_genes = set(random_data[gene_col].unique())
    borzoi_genes = set(borzoi_data[gene_col].unique())
    common_genes = random_genes & borzoi_genes

    print(f"📊 Found {len(common_genes)} genes present in both models")

    lift_analysis = []
    for gene in common_genes:
        random_subset = random_data[random_data[gene_col] == gene]
        random_mean_pearson = random_subset['pearson'].mean()
        random_std_pearson = random_subset['pearson'].std()
        random_count = len(random_subset)

        borzoi_subset = borzoi_data[borzoi_data[gene_col] == gene]
        borzoi_mean_pearson = borzoi_subset['pearson'].mean()
        borzoi_std_pearson = borzoi_subset['pearson'].std()
        borzoi_count = len(borzoi_subset)

        lift = borzoi_mean_pearson - random_mean_pearson
        relative_improvement = (lift / random_mean_pearson) * 100 if random_mean_pearson > 0 else 0

        # Only include genes with reasonable sample sizes
        if random_count >= 3 and borzoi_count >= 3:
            lift_analysis.append({
                'gene': gene,
                'random_mean': random_mean_pearson,
                'random_std': random_std_pearson,
                'random_count': random_count,
                'borzoi_mean': borzoi_mean_pearson,
                'borzoi_std': borzoi_std_pearson,
                'borzoi_count': borzoi_count,
                'lift': lift,
                'relative_improvement': relative_improvement,
                'total_samples': random_count + borzoi_count
            })

    if not lift_analysis:
        print("❌ No genes with sufficient samples for comparison")
        return None

    lift_df = pd.DataFrame(lift_analysis)

    # Merge with ad.var (by index or column)
    if ad_var.index.name == 'gene' or ad_var.index.equals(lift_df['gene']):
        annotated_df = lift_df.merge(ad_var, left_on='gene', right_index=True, how='left')
    else:
        annotated_df = lift_df.merge(ad_var, left_on='gene', right_on='gene', how='left')

    # Plotting (same as before, but with annotated_df)
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Gene Performance Lift: Human Borzoi vs Random Init', 
                 fontsize=16, fontweight='bold')
    
    # 1. LIFT BY GENE (Top Left)
    ax1 = axes[0, 0]
    lift_sorted = annotated_df.sort_values('total_samples', ascending=True).tail(top_n)
    y_pos = np.arange(len(lift_sorted))
    colors = ['darkgreen' if x > 0.01 else 'green' if x > 0.005 else 'gray' if x > -0.005 else 'red' 
              for x in lift_sorted['lift']]
    bars = ax1.barh(y_pos, lift_sorted['lift'], color=colors, alpha=0.7)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([f"{gn[:20]}... (n={row['total_samples']})" if len(gn) > 20 
                         else f"{gn} (n={row['total_samples']})" 
                         for gn, (_, row) in zip(lift_sorted['gene'], lift_sorted.iterrows())], 
                        fontsize=9)
    ax1.set_xlabel('Pearson R Lift (Human Borzoi - Random)')
    ax1.set_title('Performance Lift by Gene\n(Sorted by Sample Size)')
    ax1.axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
    ax1.grid(True, alpha=0.3)
    for i, (bar, row) in enumerate(zip(bars, lift_sorted.itertuples())):
        if abs(bar.get_width()) > 0.003:
            ax1.text(bar.get_width() + (0.002 if bar.get_width() > 0 else -0.005), 
                     bar.get_y() + bar.get_height()/2, 
                     f'{row.lift:+.3f}', ha='left' if bar.get_width() > 0 else 'right', 
                     va='center', fontsize=8, fontweight='bold')
    
    # 2. LIFT DISTRIBUTION (Top Right)
    ax2 = axes[0, 1]
    ax2.hist(annotated_df['lift'], bins=20, alpha=0.7, edgecolor='black', color='skyblue')
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No improvement')
    ax2.axvline(x=annotated_df['lift'].mean(), color='green', linestyle='-', linewidth=2, 
                label=f'Mean lift: {annotated_df["lift"].mean():.3f}')
    ax2.set_xlabel('Pearson R Lift')
    ax2.set_ylabel('Number of Genes')
    ax2.set_title('Distribution of Performance Lift')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    n_improved = (annotated_df['lift'] > 0).sum()
    n_total = len(annotated_df)
    pct_improved = (n_improved / n_total) * 100
    ax2.text(0.05, 0.95, f'Genes improved: {n_improved}/{n_total} ({pct_improved:.1f}%)', 
             transform=ax2.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    
    # 3. SCATTER PLOT: Random vs Borzoi Performance (Bottom Left)
    ax3 = axes[1, 0]
    sizes = annotated_df['total_samples'] * 3
    colors_scatter = ['green' if x > 0 else 'red' for x in annotated_df['lift']]
    scatter = ax3.scatter(annotated_df['random_mean'], annotated_df['borzoi_mean'], 
                          s=sizes, c=colors_scatter, alpha=0.6, edgecolors='black', linewidth=0.5)
    min_val = min(annotated_df['random_mean'].min(), annotated_df['borzoi_mean'].min())
    max_val = max(annotated_df['random_mean'].max(), annotated_df['borzoi_mean'].max())
    ax3.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Equal performance')
    ax3.set_xlabel('Random Init Mean Pearson R')
    ax3.set_ylabel('Human Borzoi Mean Pearson R')
    ax3.set_title('Performance Comparison\n(Point size = sample count)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    from scipy import stats
    corr_r, corr_p = stats.pearsonr(annotated_df['random_mean'], annotated_df['borzoi_mean'])
    ax3.text(0.05, 0.95, f'Correlation: r={corr_r:.3f} (p={corr_p:.3f})', 
             transform=ax3.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 4. TOP IMPROVEMENTS TABLE (Bottom Right)
    ax4 = axes[1, 1]
    ax4.axis('off')
    top_improvements = annotated_df.nlargest(8, 'lift')[['gene', 'lift', 'relative_improvement', 
                                                    'random_mean', 'borzoi_mean', 'total_samples']]
    table_data = []
    for _, row in top_improvements.iterrows():
        table_data.append([
            row['gene'][:15] + '...' if len(row['gene']) > 15 else row['gene'],
            f"{row['lift']:+.3f}",
            f"{row['relative_improvement']:+.1f}%",
            f"{row['random_mean']:.3f}",
            f"{row['borzoi_mean']:.3f}",
            f"{row['total_samples']}"
        ])
    if table_data:
        table = ax4.table(cellText=table_data,
                          colLabels=['Gene', 'Lift', 'Rel. Imp.', 'Random', 'Borzoi', 'n'],
                          cellLoc='center',
                          loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.2, 1.5)
        ax4.set_title('Top Improvements', pad=20, fontsize=12, fontweight='bold')
        for i in range(len(table_data)):
            lift_val = float(table_data[i][1])
            if lift_val > 0.01:
                table[(i+1, 1)].set_facecolor('lightgreen')
            elif lift_val > 0.005:
                table[(i+1, 1)].set_facecolor('lightyellow')

    plt.tight_layout()
    plt.show()

    # Print detailed summary
    print(f"\n📈 LIFT ANALYSIS SUMMARY:")
    print(f"   • Total genes analyzed: {len(annotated_df)}")
    print(f"   • Genes with improved performance: {n_improved} ({pct_improved:.1f}%)")
    print(f"   • Mean lift across all genes: {annotated_df['lift'].mean():.4f}")
    print(f"   • Median lift: {annotated_df['lift'].median():.4f}")
    print(f"   • Standard deviation of lift: {annotated_df['lift'].std():.4f}")

    print(f"\n🏆 TOP 5 IMPROVEMENTS:")
    top_5 = annotated_df.nlargest(5, 'lift')
    for i, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"   {i}. {row['gene']}: +{row['lift']:.3f} ({row['relative_improvement']:+.1f}%)")
        print(f"      Random: {row['random_mean']:.3f} → Borzoi: {row['borzoi_mean']:.3f} (n={row['total_samples']})")

    print(f"\n📉 BOTTOM 3 (if any declines):")
    bottom_3 = annotated_df.nsmallest(3, 'lift')
    for i, (_, row) in enumerate(bottom_3.iterrows(), 1):
        print(f"   {i}. {row['gene']}: {row['lift']:+.3f} ({row['relative_improvement']:+.1f}%)")
        print(f"      Random: {row['random_mean']:.3f} → Borzoi: {row['borzoi_mean']:.3f} (n={row['total_samples']})")

    return fig, annotated_df


In [ ]:
import pandas as pd

# Combine gene-level (var) metrics across replicates for each model type
gene_dfs = {}
for model_type in ['Random Init', 'Human Borzoi']:
    model_reps = aggregated_data[model_type]['replicates']
    dfs = []
    for adata, rep_name in model_reps:
        df = adata.var.copy()
        df['replicate'] = rep_name
        df['model_type'] = model_type
        dfs.append(df)
    gene_dfs[model_type] = pd.concat(dfs, axis=0, ignore_index=True)

# For compatibility with analyze_gene_lift, build a new structure
agg_gene_data = {
    'Random Init': {'combined_df': gene_dfs['Random Init']},
    'Human Borzoi': {'combined_df': gene_dfs['Human Borzoi']}
}


In [ ]:
# ad_var is your AnnData .var, indexed by gene, with annotation columns
fig, gene_lift_df = analyze_gene_lift(
    agg_gene_data,   # prepared above
    ad_decima.var,          # your annotation DataFrame, e.g. ad.var or similar
    gene_col='gene'  # or whatever the gene column is called in your dataframes
)
